**Learning for Fun**

This is just me messing around with other datasets to see how the model will react. I am trying to gauge how different types of data as well as data sets will interact with one another.

First dataset expirment is with Soil data from SoilGrid:

Hengl, T., de Jesus, J. M., Heuvelink, G. B. M., Gonzalez, M. R., Kilibarda, M., Blagotić, A., et al. (2017). SoilGrids250m: Global gridded soil information based on machine learning. PLOS ONE, 12(2). https://doi.org/10.1371/journal.pone.0169748

[SoilGrid](https://soilgrids.org/)


Additional Water Data
Of course — here are **clean, professional citations** you can drop directly into your report, README, or methods section for both datasets.

I’ll give you **APA-style (best for reports)** and **short dataset citations (great for GitHub / Kaggle)**.

---

# 1. GLORICH Dataset Citation

GLORICH Global River Chemistry Database

### APA Citation

Hartmann, J., Lauerwald, R., & Moosdorf, N. (2019). *GLORICH – Global River Chemistry Database* [Dataset]. PANGAEA. [https://doi.org/10.1594/PANGAEA.902360](https://doi.org/10.1594/PANGAEA.902360) ([PANGAEA DOI Resolver][1])

### Supporting Paper Citation

Hartmann, J., Lauerwald, R., & Moosdorf, N. (2014). *A brief overview of the GLObal RIver CHemistry Database (GLORICH).* Procedia Earth and Planetary Science, 10, 23–27. [https://doi.org/10.1016/j.proeps.2014.08.005](https://doi.org/10.1016/j.proeps.2014.08.005) ([ESSD][2])

### Short Version (README friendly)

> River chemistry data were obtained from the **GLORICH Global River Chemistry Database**, compiled by Hartmann, Lauerwald, and Moosdorf and distributed through PANGAEA. ([PANGAEA DOI Resolver][1])

The database contains **over 1.27 million measurements from more than 18,000 sampling locations worldwide**, including nutrients, conductivity, and other hydrochemical parameters. ([PANGAEA DOI Resolver][1])

---

### Short Version (README friendly)

> Hydro-environmental basin attributes were obtained from **HydroATLAS**, a global dataset of watershed characteristics derived from the HydroSHEDS framework. ([HydroSHEDS][3])


```
DATA SOURCES

• River chemistry data: GLORICH Global River Chemistry Database (Hartmann et al., 2019)
• Soil chemistry: SoilGrids (ISRIC)
• Climate data: TerraClimate
• Satellite reflectance: Landsat
• Land cover: ESA WorldCover
```


[1]: https://doi.pangaea.de/10.1594/PANGAEA.902360?utm_source=chatgpt.com "Hartmann, J et al. (2019): GLORICH - Global river chemistry database"
[2]: https://essd.copernicus.org/articles/14/4667/2022/?utm_source=chatgpt.com "ESSD - The Surface Water Chemistry (SWatCh) database: a standardized global database of water chemistry to facilitate large-sample hydrological research"
[3]: https://www.hydrosheds.org/hydroatlas?utm_source=chatgpt.com "HydroATLAS"


In [ ]:
# =============================================================
# RANDOM FOREST — PHASE 5 + WOSIS + GLORICH
#
# Built on the locked Phase 5 architecture (leaderboard: 0.3049).
# Zero changes to model hyperparameters, CV setup, or pipeline.
# Two new external data sources added as additional feature blocks:
#
#   WoSIS Soil Properties (9 features):
#     Clay content, Organic Carbon, Soil pH — surface layers only
#     (upper_depth ≤ 30cm). Spatially stable; no leakage risk.
#     Key signal: soil_phaq → TA (r=+0.29), soil_clay_std → DRP (r=+0.25)
#
#   GloRiCh River Chemistry (15 features):
#     416,837 SA observations across 1,304 unique stations.
#     Median nearest-station distance ~750m — essentially co-located.
#     Key signal: glorich_TA → TA (r=+0.82), glorich_DIP → DRP (r=+0.47)
#     Also: glorich_Catch_Slope → DRP (r=-0.35), GLC_Forest → DRP (r=-0.27)
#
# LOCKED FROM PHASE 5 (unchanged):
#   LOG_DRP_TARGET = False
#   NWU_K = 5, N_BASIN_CLUSTERS = 12
#   TA/EC: max_depth=12, min_samples_leaf=4
#   DRP:   max_depth=10, min_samples_leaf=6
#   Pruned: green, basin_n_stations, doy_sin, wet_phase_cos,
#           month_sin, month_cos
#
# GOOGLE COLAB USAGE:
#   Upload all CSVs to Colab, then run. File names must match
#   exactly as listed in Section 1 (LOAD DATA).
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


# =============================================================
# CONFIGURATION  (all locked from Phase 5)
# =============================================================
LOG_DRP_TARGET   = False
N_BASIN_CLUSTERS = 12
NWU_K            = 5
IDW_EPS          = 1e-6
LEAF_TA_EC       = 4
LEAF_DRP         = 6    # Best from Phase 5 leaf sweep

WOSIS_K          = 5    # Neighbours for WoSIS KD-tree
GLORICH_K        = 5    # Neighbours for GloRiCh KD-tree
SURFACE_DEPTH    = 30   # WoSIS: only layers where upper_depth <= this (cm)


# =============================================================
# 1. LOAD DATA
# =============================================================
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

base_df    = pd.read_csv('water_quality_training_dataset.csv')
partner_df = pd.read_csv('water_training_added_features.csv')
rain_df    = pd.read_csv('nasa_rain_features.csv')
lc_df      = pd.read_csv('esa_landcover_features.csv')
landsat_df = pd.read_csv('landsat_features_training.csv')
terra_df   = pd.read_csv('terraclimate_features_training.csv')
nwu_df     = pd.read_csv('nwu_water_quality_clean.csv')
test_df    = pd.read_csv('submission_template.csv')

# New external datasets
clay_df    = pd.read_csv('wosis_latest_clay.csv')
orgc_df    = pd.read_csv('wosis_latest_orgc.csv')
phaq_df    = pd.read_csv('wosis_latest_phaq.csv')
glorich_df = pd.read_csv('glorich_cleaned_chemistry_dataset.csv')

print(f"Training rows:   {len(base_df):,}")
print(f"Submission rows: {len(test_df):,}")
print(f"NWU stations:    {len(nwu_df):,}")
print(f"GloRiCh rows:    {len(glorich_df):,}")
print(f"WoSIS clay:      {len(clay_df):,}")
print(f"WoSIS orgc:      {len(orgc_df):,}")
print(f"WoSIS phaq:      {len(phaq_df):,}")


# =============================================================
# 2. DATE NORMALISATION
# =============================================================
def normalise_date(df):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(
        df['Sample Date'], format='mixed', dayfirst=True
    ).dt.strftime('%Y-%m-%d')
    return df

base_df    = normalise_date(base_df)
partner_df = normalise_date(partner_df)
rain_df    = normalise_date(rain_df)
test_df    = normalise_date(test_df)
landsat_df = normalise_date(landsat_df)
terra_df   = normalise_date(terra_df)
nwu_df     = normalise_date(nwu_df)

JOIN_KEYS = ['Latitude', 'Longitude', 'Sample Date']


# =============================================================
# 3. SPATIAL FILL HELPER  (unchanged from Phase 5)
# =============================================================
def spatial_fill(base, feature_df, join_keys=JOIN_KEYS):
    already_in_base = set(base.columns) - set(join_keys)
    feat_cols = [
        c for c in feature_df.columns
        if c not in join_keys and c not in already_in_base
    ]
    if not feat_cols:
        print("  No new feature columns to add — skipping merge.")
        return base

    feature_slim = feature_df[join_keys + feat_cols].copy()
    merged       = base.merge(feature_slim, on=join_keys, how='left')
    null_mask    = merged[feat_cols].isnull().any(axis=1)
    n_missing    = null_mask.sum()

    if n_missing == 0:
        print(f"  Exact join: 100% match ({len(base):,} rows)")
        return merged

    print(f"  Exact join: {len(base) - n_missing:,}/{len(base):,} rows matched "
          f"— {n_missing:,} rows need spatial fill")

    stat_medians = (
        feature_df
        .groupby(['Latitude', 'Longitude'])[feat_cols]
        .median()
        .reset_index()
    )
    tree           = cKDTree(stat_medians[['Latitude', 'Longitude']].values)
    missing_coords = base.loc[null_mask, ['Latitude', 'Longitude']].values
    _, nn_idxs     = tree.query(missing_coords, k=1)
    nearest_feats  = stat_medians[feat_cols].iloc[nn_idxs].reset_index(drop=True)
    merged_idx     = merged.index[null_mask]
    for col in feat_cols:
        merged.loc[merged_idx, col] = nearest_feats[col].values

    remaining_null = merged[feat_cols].isnull().any(axis=1).sum()
    print(f"  After spatial fill: {remaining_null} rows still null (expected 0)")
    return merged


# =============================================================
# 4. NWU SPATIAL CONTEXT FEATURES  (unchanged from Phase 5)
# =============================================================
def build_nwu_tree(nwu_df):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            nwu_TA_med  = ('Total Alkalinity',              'median'),
            nwu_TA_std  = ('Total Alkalinity',              'std'),
            nwu_EC_med  = ('Electrical Conductance',        'median'),
            nwu_EC_std  = ('Electrical Conductance',        'std'),
            nwu_DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
            nwu_DRP_std = ('Dissolved Reactive Phosphorus', 'std'),
        )
        .reset_index()
        .fillna(0)
    )
    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  NWU KD-tree: {len(stations)} unique stations")
    return stations, tree


def add_nwu_features(df, stations, tree, k=NWU_K):
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=k)

    k1      = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    arr_TA  = stations['nwu_TA_med'].values[idxs]
    arr_EC  = stations['nwu_EC_med'].values[idxs]
    arr_DRP = stations['nwu_DRP_med'].values[idxs]

    weights    = 1.0 / (dists + IDW_EPS)
    weight_sum = weights.sum(axis=1, keepdims=True)
    w_norm     = weights / weight_sum

    result = df.copy()
    result['nwu_k1_TA']       = k1['nwu_TA_med'].values
    result['nwu_k1_EC']       = k1['nwu_EC_med'].values
    result['nwu_k1_DRP']      = k1['nwu_DRP_med'].values
    result['nwu_k5_TA_idw']   = (arr_TA  * w_norm).sum(axis=1)
    result['nwu_k5_EC_idw']   = (arr_EC  * w_norm).sum(axis=1)
    result['nwu_k5_DRP_idw']  = (arr_DRP * w_norm).sum(axis=1)
    result['nwu_k5_TA_std']   = arr_TA.std(axis=1)
    result['nwu_k5_EC_std']   = arr_EC.std(axis=1)
    result['nwu_k5_DRP_std']  = arr_DRP.std(axis=1)
    result['nwu_k1_dist_deg'] = dists[:, 0]

    return result


# =============================================================
# 5. BASIN CHEMISTRY FEATURES  (unchanged from Phase 5)
# =============================================================
def build_basin_model(nwu_df, n_clusters=N_BASIN_CLUSTERS):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            TA_med  = ('Total Alkalinity',              'median'),
            EC_med  = ('Electrical Conductance',        'median'),
            DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
        )
        .reset_index()
    )
    coords = stations[['Latitude', 'Longitude']].values
    km     = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    stations['basin'] = km.fit_predict(coords)

    basin_profiles = (
        stations
        .groupby('basin')
        .agg(
            basin_TA_med     = ('TA_med',  'median'),
            basin_TA_std     = ('TA_med',  'std'),
            basin_EC_med     = ('EC_med',  'median'),
            basin_EC_std     = ('EC_med',  'std'),
            basin_DRP_med    = ('DRP_med', 'median'),
            basin_DRP_std    = ('DRP_med', 'std'),
        )
        .reset_index()
        .fillna(0)
    )
    print(f"  Basin model: {n_clusters} clusters over {len(stations)} NWU stations")
    return stations, basin_profiles, km, coords


def add_basin_features(df, station_coords, basin_profiles, km):
    coords       = df[['Latitude', 'Longitude']].values
    basin_labels = km.predict(coords)
    profile_lookup = basin_profiles.set_index('basin')
    row_profiles   = profile_lookup.loc[basin_labels].reset_index(drop=True)
    result = df.copy()
    for col in profile_lookup.columns:
        result[col] = row_profiles[col].values
    return result


# =============================================================
# 6. WOSIS SOIL FEATURES  (NEW)
#
# Physical mechanisms:
#   clay  → high clay content increases P adsorption, lowering DRP
#   orgc  → organic carbon competes with phosphate for sorption
#   phaq  → soil pH controls P solubility AND carbonate equilibrium
#            (most direct physical link to Total Alkalinity)
#
# Three features per property (same IDW pattern as NWU):
#   _k1   = nearest soil profile value
#   _idw  = IDW-weighted mean over k=5 nearest
#   _std  = spatial variability over k=5 nearest
#           (soil_clay_std was highest DRP correlate at r=+0.25)
#
# Verified: 0 nulls on train and test, all 7,500+ profiles
# are South Africa only, median distance ~22km to both
# train and test — identical proximity, no leakage.
# =============================================================
def _build_wosis_tree(df, prop_name):
    """Aggregate a WoSIS dataframe to surface-layer unique locations."""
    surf = df[df['upper_depth'] <= SURFACE_DEPTH].copy()
    agg  = (
        surf
        .groupby(['X', 'Y'])['value_avg']
        .mean()
        .reset_index()
        .rename(columns={'value_avg': prop_name})
        .dropna(subset=[prop_name])
    )
    tree = cKDTree(agg[['Y', 'X']].values)  # (lat, lon) order
    print(f"  WoSIS {prop_name}: {len(agg):,} unique surface locations")
    return agg, tree


def build_wosis_trees(clay_df, orgc_df, phaq_df):
    print("\n[WoSIS soil features — building KD-trees]")
    return {
        'clay': _build_wosis_tree(clay_df, 'clay'),
        'orgc': _build_wosis_tree(orgc_df, 'orgc'),
        'phaq': _build_wosis_tree(phaq_df, 'phaq'),
    }


def add_wosis_features(df, wosis_trees):
    result = df.copy()
    for prop_name, (profiles, tree) in wosis_trees.items():
        coords      = result[['Latitude', 'Longitude']].values
        dists, idxs = tree.query(coords, k=WOSIS_K)
        values      = profiles[prop_name].values[idxs]
        weights     = 1.0 / (dists + IDW_EPS)
        w_norm      = weights / weights.sum(axis=1, keepdims=True)

        result[f'soil_{prop_name}_k1']  = values[:, 0]
        result[f'soil_{prop_name}_idw'] = (values * w_norm).sum(axis=1)
        result[f'soil_{prop_name}_std'] = values.std(axis=1)
    return result


# =============================================================
# 7. GLORICH RIVER CHEMISTRY FEATURES  (NEW)
#
# 416,837 SA observations, 1,304 unique stations.
# Median nearest-station distance ~750m — essentially co-located.
# Train and test have identical proximity → no leakage advantage.
#
# Chemistry (IDW k=5, same pattern as NWU):
#   glorich_TA_*   — independent TA anchor (r=+0.82 with target TA)
#   glorich_EC_*   — independent EC anchor (r=+0.57 with target EC)
#   glorich_DIP_*  — Dissolved Inorganic P, best DRP proxy (r=+0.47)
#
# Catchment covariates (nearest station, static per catchment):
#   glorich_Catch_Slope  — r=-0.35 with DRP (steeper = faster runoff)
#   glorich_GLC_Forest   — r=-0.27 with DRP (forest retains P)
#   glorich_Altitude, Soil_pH, SOC, GLC_Managed
# =============================================================
def build_glorich_tree(glorich_df):
    print("\n[GloRiCh — filtering to South Africa and aggregating]")
    sa = glorich_df[glorich_df['Country'] == 'South Africa'].copy()
    print(f"  SA observations: {len(sa):,}")

    stations = (
        sa.groupby(['Latitude', 'Longitude'])
        .agg(
            glorich_TA_med      = ('Total_Alkalinity',      'median'),
            glorich_TA_std      = ('Total_Alkalinity',      'std'),
            glorich_EC_med      = ('EC',                    'median'),
            glorich_EC_std      = ('EC',                    'std'),
            glorich_DIP_med     = ('Dissolved_Inorganic_P', 'median'),
            glorich_DIP_std     = ('Dissolved_Inorganic_P', 'std'),
            glorich_Soil_pH     = ('Soil_pH',               'median'),
            glorich_SOC         = ('SOC',                   'median'),
            glorich_Altitude    = ('Altitude',              'median'),
            glorich_Catch_Slope = ('Catch_Slope',           'median'),
            glorich_GLC_Forest  = ('GLC_Forest',            'median'),
            glorich_GLC_Managed = ('GLC_Managed',           'median'),
        )
        .reset_index()
    )

    for col in stations.columns:
        if stations[col].isnull().any():
            stations[col] = stations[col].fillna(stations[col].median())

    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  GloRiCh KD-tree: {len(stations):,} unique SA stations")
    return stations, tree


def add_glorich_features(df, stations, tree):
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=GLORICH_K)

    result = df.copy()

    # Chemistry: IDW features for TA, EC, DIP
    for med_col, k1_col, idw_col, std_col in [
        ('glorich_TA_med',  'glorich_TA_k1',  'glorich_TA_idw',  'glorich_TA_std_feat'),
        ('glorich_EC_med',  'glorich_EC_k1',  'glorich_EC_idw',  'glorich_EC_std_feat'),
        ('glorich_DIP_med', 'glorich_DIP_k1', 'glorich_DIP_idw', 'glorich_DIP_std_feat'),
    ]:
        values  = stations[med_col].values[idxs]
        weights = 1.0 / (dists + IDW_EPS)
        w_norm  = weights / weights.sum(axis=1, keepdims=True)

        result[k1_col]  = values[:, 0]
        result[idw_col] = (values * w_norm).sum(axis=1)
        result[std_col] = values.std(axis=1)

    # Catchment covariates: nearest station only
    nn = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    for col in ['glorich_Soil_pH', 'glorich_SOC', 'glorich_Altitude',
                'glorich_Catch_Slope', 'glorich_GLC_Forest', 'glorich_GLC_Managed']:
        result[col] = nn[col].values

    return result


# =============================================================
# 8. MERGE ALL FEATURES — TRAINING
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — TRAINING DATA")
print("=" * 60)

print("\n[partner / flow features]")
train_df = spatial_fill(base_df, partner_df)

print("\n[rain features]")
train_df = spatial_fill(train_df, rain_df)

print("\n[land cover features]")
train_df = train_df.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
train_df = train_df.merge(landsat_df, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
train_df = train_df.merge(terra_df, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features]")
nwu_stations, nwu_tree = build_nwu_tree(nwu_df)
train_df = add_nwu_features(train_df, nwu_stations, nwu_tree)

print("\n[Basin chemistry features]")
station_df, basin_profiles, km_basin, station_coords = build_basin_model(nwu_df)
train_df = add_basin_features(train_df, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
wosis_trees = build_wosis_trees(clay_df, orgc_df, phaq_df)
train_df = add_wosis_features(train_df, wosis_trees)

print("\n[GloRiCh river chemistry features]")
glorich_stations, glorich_tree = build_glorich_tree(glorich_df)
train_df = add_glorich_features(train_df, glorich_stations, glorich_tree)

print(f"\nFinal training shape: {train_df.shape}")


# =============================================================
# 9. MERGE ALL FEATURES — SUBMISSION
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — SUBMISSION DATA")
print("=" * 60)

landsat_val = normalise_date(pd.read_csv('landsat_features_validation.csv'))
terra_val   = normalise_date(pd.read_csv('terraclimate_features_validation.csv'))

print("\n[partner / flow features]")
test_df2 = spatial_fill(test_df, partner_df)

print("\n[rain features]")
test_df2 = spatial_fill(test_df2, rain_df)

print("\n[land cover features]")
test_df2 = test_df2.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
test_df2 = test_df2.merge(landsat_val, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
test_df2 = test_df2.merge(terra_val, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features]")
test_df2 = add_nwu_features(test_df2, nwu_stations, nwu_tree)

print("\n[Basin chemistry features]")
test_df2 = add_basin_features(test_df2, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
test_df2 = add_wosis_features(test_df2, wosis_trees)

print("\n[GloRiCh river chemistry features]")
test_df2 = add_glorich_features(test_df2, glorich_stations, glorich_tree)

print(f"\nFinal submission shape: {test_df2.shape}")


# =============================================================
# 10. TEMPORAL FEATURES  (unchanged from Phase 5)
# =============================================================
def add_temporal(df):
    df    = df.copy()
    dates = pd.to_datetime(df['Sample Date'])
    df['DayOfYear']           = dates.dt.dayofyear
    df['Month']               = dates.dt.month
    df['days_since_wet_start'] = (df['DayOfYear'] - 274) % 365
    return df

train_df = add_temporal(train_df)
test_df2 = add_temporal(test_df2)


# =============================================================
# 11. PHYSICS-BASED INTERACTION FEATURES  (unchanged from Phase 5)
# =============================================================
def add_interactions(df):
    df = df.copy()
    if 'rain_30d_sum' in df.columns and 'temp_7d_mean' in df.columns:
        df['rain_temp_30d'] = df['rain_30d_sum'] * df['temp_7d_mean']
    if 'rain_30d_mean' in df.columns and 'rh_7d_mean' in df.columns:
        df['rain_rh_30d'] = df['rain_30d_mean'] * df['rh_7d_mean']
    if 'rain_30d_sum' in df.columns and 'rain_7d_mean' in df.columns:
        df['rain_pulse_ratio'] = df['rain_7d_mean'] / (df['rain_30d_sum'] / 30 + 1e-6)
    if 'pet' in df.columns and 'rain_30d_mean' in df.columns:
        df['pet_rain_ratio'] = df['pet'] / (df['rain_30d_mean'] + 1e-6)
    if 'NDMI' in df.columns and 'rain_30d_sum' in df.columns:
        df['ndmi_rain'] = df['NDMI'] * df['rain_30d_sum']
    return df

train_df = add_interactions(train_df)
test_df2 = add_interactions(test_df2)


# =============================================================
# 12. RAINFALL PULSE FEATURES  (unchanged from Phase 5)
# =============================================================
def add_rainfall_pulse_features(df):
    df = df.copy()
    if 'rain_7d_mean' in df.columns:
        df['rain_7d_max'] = df['rain_7d_mean'] * 2.5
    if 'rain_7d_mean' in df.columns and 'rain_30d_mean' in df.columns:
        df['rain_dry_spell_ratio'] = (
            df['rain_30d_mean'] / (df['rain_7d_mean'] + 1e-6)
        ).clip(upper=50)
    return df

train_df = add_rainfall_pulse_features(train_df)
test_df2 = add_rainfall_pulse_features(test_df2)


# =============================================================
# 13. FEATURE COLUMNS
# Phase 5 base features (unchanged) + WoSIS + GloRiCh blocks
# =============================================================
BASELINE_FEATURES = [
    'Longitude', 'Latitude',
    'rh_7d_mean', 'distance_to_station_km',
    'rain_30d_mean', 'rain_30d_sum',
    'temp_7d_mean',
]

EY_FEATURES = [
    'nir', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'pet',
]

TEMPORAL_FEATURES = [
    'DayOfYear',
    'days_since_wet_start',
]

INTERACTION_FEATURES = [
    'rain_temp_30d', 'rain_rh_30d', 'rain_pulse_ratio',
    'pet_rain_ratio', 'ndmi_rain',
]

PULSE_FEATURES = [
    'rain_7d_max',
    'rain_dry_spell_ratio',
]

NWU_FEATURES = [
    'nwu_k1_TA',  'nwu_k1_EC',  'nwu_k1_DRP',
    'nwu_k5_TA_idw', 'nwu_k5_EC_idw', 'nwu_k5_DRP_idw',
    'nwu_k5_TA_std', 'nwu_k5_EC_std', 'nwu_k5_DRP_std',
    'nwu_k1_dist_deg',
]

BASIN_FEATURES = [
    'basin_TA_med', 'basin_TA_std',
    'basin_EC_med', 'basin_EC_std',
    'basin_DRP_med', 'basin_DRP_std',
]

# --- NEW: WoSIS soil properties ---
WOSIS_FEATURES = [
    'soil_clay_k1',  'soil_clay_idw',  'soil_clay_std',
    'soil_orgc_k1',  'soil_orgc_idw',  'soil_orgc_std',
    'soil_phaq_k1',  'soil_phaq_idw',  'soil_phaq_std',
]

# --- NEW: GloRiCh river chemistry & catchment covariates ---
GLORICH_FEATURES = [
    'glorich_TA_k1',   'glorich_TA_idw',   'glorich_TA_std_feat',
    'glorich_EC_k1',   'glorich_EC_idw',   'glorich_EC_std_feat',
    'glorich_DIP_k1',  'glorich_DIP_idw',  'glorich_DIP_std_feat',
    'glorich_Soil_pH', 'glorich_SOC',      'glorich_Altitude',
    'glorich_Catch_Slope', 'glorich_GLC_Forest', 'glorich_GLC_Managed',
]

FEATURE_COLS = [
    c for c in
    BASELINE_FEATURES + EY_FEATURES + TEMPORAL_FEATURES
    + INTERACTION_FEATURES + PULSE_FEATURES
    + NWU_FEATURES + BASIN_FEATURES
    + WOSIS_FEATURES + GLORICH_FEATURES
    if c in train_df.columns
]

print(f"\n{'=' * 60}")
print("FEATURE SUMMARY")
print(f"{'=' * 60}")
print(f"Total features: {len(FEATURE_COLS)}")
print(f"  Baseline:        {sum(c in train_df.columns for c in BASELINE_FEATURES)}")
print(f"  EY satellite:    {sum(c in train_df.columns for c in EY_FEATURES)}")
print(f"  Temporal:        {sum(c in train_df.columns for c in TEMPORAL_FEATURES)}")
print(f"  Interaction:     {sum(c in train_df.columns for c in INTERACTION_FEATURES)}")
print(f"  Pulse:           {sum(c in train_df.columns for c in PULSE_FEATURES)}")
print(f"  NWU context:     {sum(c in train_df.columns for c in NWU_FEATURES)}")
print(f"  Basin context:   {sum(c in train_df.columns for c in BASIN_FEATURES)}")
print(f"  WoSIS soil:      {sum(c in train_df.columns for c in WOSIS_FEATURES)}  (NEW)")
print(f"  GloRiCh:         {sum(c in train_df.columns for c in GLORICH_FEATURES)}  (NEW)")


# =============================================================
# 14. PREPARE DATA
# =============================================================
def resolve_target_col(df, col):
    if col in df.columns:        return df[col]
    if col + '_x' in df.columns: return df[col + '_x']
    raise KeyError(f"Target column '{col}' not found.")

X = train_df[FEATURE_COLS].copy().fillna(train_df[FEATURE_COLS].median())
train_medians = X.median()

y_TA  = resolve_target_col(train_df, 'Total Alkalinity')
y_EC  = resolve_target_col(train_df, 'Electrical Conductance')
y_DRP = resolve_target_col(train_df, 'Dissolved Reactive Phosphorus')


# =============================================================
# 15. CROSS VALIDATION
# =============================================================
cv = KFold(n_splits=5, shuffle=True, random_state=42)


# =============================================================
# 16. MODEL FACTORY  (unchanged from Phase 5)
# =============================================================
def make_pipe(depth, min_leaf):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            n_estimators     = 500,
            max_depth        = depth,
            min_samples_leaf = min_leaf,
            max_features     = 'sqrt',
            bootstrap        = True,
            random_state     = 42,
            n_jobs           = -1,
        ))
    ])


# =============================================================
# 17. TRAIN TA AND EC
# =============================================================
MODEL_CONFIGS = {
    'Total Alkalinity':       dict(y=y_TA,  depth=12, leaf=LEAF_TA_EC),
    'Electrical Conductance': dict(y=y_EC,  depth=12, leaf=LEAF_TA_EC),
}

models  = {}
results = []

for name, cfg in MODEL_CONFIGS.items():
    y_target = cfg['y']
    depth    = cfg['depth']
    leaf     = cfg['leaf']

    print("\n" + "=" * 60)
    print(f"Training: {name}  (max_depth={depth}, min_samples_leaf={leaf})")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_target, test_size=0.3, random_state=42
    )

    pipe      = make_pipe(depth, leaf)
    cv_scores = cross_val_score(pipe, X, y_target, cv=cv, scoring='r2', n_jobs=-1)

    pipe.fit(X_train, y_train)
    r2_train = r2_score(y_train, pipe.predict(X_train))
    r2_test  = r2_score(y_test,  pipe.predict(X_test))
    rmse     = np.sqrt(mean_squared_error(y_test, pipe.predict(X_test)))
    gap      = r2_train - r2_test

    print(f"KFold CV R²:    {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  "
          f"(Phase 5 ref: TA=0.8551, EC=0.8584)")
    print(f"Train R²:       {r2_train:.4f}")
    print(f"Test  R²:       {r2_test:.4f}")
    print(f"Train/Test Gap: {gap:.4f}  (target: < 0.10)")
    print(f"RMSE:           {rmse:.4f}")

    models[name] = pipe
    results.append({
        'Parameter':      name,
        'KFold_CV_R2':    round(cv_scores.mean(), 4),
        'CV_Std':         round(cv_scores.std(),  4),
        'R2_Train':       round(r2_train, 4),
        'R2_Test':        round(r2_test,  4),
        'Train_Test_Gap': round(gap, 4),
        'RMSE':           round(rmse, 4),
    })


# =============================================================
# 18. TRAIN DRP
# =============================================================
print("\n" + "=" * 60)
print(f"Training: Dissolved Reactive Phosphorus  "
      f"(max_depth=10, min_samples_leaf={LEAF_DRP})")
print(f"Phase 5 reference CV R²: 0.6255")
print("=" * 60)

X_train_drp, X_test_drp, y_train_drp, y_test_drp = train_test_split(
    X, y_DRP, test_size=0.3, random_state=42
)

drp_pipe  = make_pipe(depth=10, min_leaf=LEAF_DRP)
cv_scores = cross_val_score(drp_pipe, X, y_DRP, cv=cv, scoring='r2', n_jobs=-1)

drp_pipe.fit(X_train_drp, y_train_drp)
r2_train = r2_score(y_train_drp, drp_pipe.predict(X_train_drp))
r2_test  = r2_score(y_test_drp,  drp_pipe.predict(X_test_drp))
rmse     = np.sqrt(mean_squared_error(y_test_drp, drp_pipe.predict(X_test_drp)))
gap      = r2_train - r2_test
delta    = cv_scores.mean() - 0.6255

print(f"KFold CV R²:    {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  "
      f"(delta vs Phase 5: {delta:+.4f})")
print(f"Train R²:       {r2_train:.4f}")
print(f"Test  R²:       {r2_test:.4f}")
print(f"Train/Test Gap: {gap:.4f}  (target: < 0.10)")
print(f"RMSE:           {rmse:.4f}")

models['Dissolved Reactive Phosphorus'] = drp_pipe
results.append({
    'Parameter':      'Dissolved Reactive Phosphorus',
    'KFold_CV_R2':    round(cv_scores.mean(), 4),
    'CV_Std':         round(cv_scores.std(),  4),
    'R2_Train':       round(r2_train, 4),
    'R2_Test':        round(r2_test,  4),
    'Train_Test_Gap': round(gap, 4),
    'RMSE':           round(rmse, 4),
})


# =============================================================
# 19. FINAL SUMMARY
# =============================================================
results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("FINAL SUMMARY  (Phase 5 reference: TA=0.8551, EC=0.8584, DRP=0.6255)")
print("=" * 60)
print(results_df.to_string(index=False))


# =============================================================
# 20. FEATURE IMPORTANCE
# =============================================================
print("\n" + "=" * 60)
print("FEATURE IMPORTANCES (top 20 per target)")
print("=" * 60)

importance_dfs = []

for name, pipe in models.items():
    rf_model   = pipe.named_steps['rf']
    imp        = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
    imp_sorted = imp.sort_values(ascending=False)

    print(f"\n--- {name} ---")
    print(imp_sorted.head(20).round(4).to_string())

    imp_df         = imp_sorted.reset_index()
    imp_df.columns = ['Feature', name]
    importance_dfs.append(imp_df.set_index('Feature'))

combined_imp = pd.concat(importance_dfs, axis=1)
combined_imp['Mean_Importance'] = combined_imp.mean(axis=1)
combined_imp = combined_imp.sort_values('Mean_Importance', ascending=False)

print("\n--- COMBINED (mean importance across all targets) ---")
print(combined_imp[['Mean_Importance']].round(4).to_string())

print("\n→ New WoSIS feature importances:")
for f in WOSIS_FEATURES:
    if f in combined_imp.index:
        drp_imp  = combined_imp.loc[f, 'Dissolved Reactive Phosphorus']
        mean_imp = combined_imp.loc[f, 'Mean_Importance']
        print(f"  {f:30s}  mean={mean_imp:.4f}  DRP={drp_imp:.4f}")

print("\n→ New GloRiCh feature importances:")
for f in GLORICH_FEATURES:
    if f in combined_imp.index:
        drp_imp  = combined_imp.loc[f, 'Dissolved Reactive Phosphorus']
        mean_imp = combined_imp.loc[f, 'Mean_Importance']
        print(f"  {f:30s}  mean={mean_imp:.4f}  DRP={drp_imp:.4f}")

print("\n→ Bottom 5 candidates to drop next iteration:")
print(combined_imp[['Mean_Importance']].tail(5).round(4).to_string())


# =============================================================
# 21. SUBMISSION
# =============================================================
X_val = test_df2[FEATURE_COLS].copy().fillna(train_medians)

submission_df = pd.DataFrame({
    'Longitude':                     test_df2['Longitude'],
    'Latitude':                      test_df2['Latitude'],
    'Sample Date':                   test_df2['Sample Date'],
    'Total Alkalinity':              models['Total Alkalinity'].predict(X_val),
    'Electrical Conductance':        models['Electrical Conductance'].predict(X_val),
    'Dissolved Reactive Phosphorus': models['Dissolved Reactive Phosphorus'].predict(X_val),
})

fname = 'submission_rf_phase5_wosis_glorich.csv'
submission_df.to_csv(fname, index=False)

print(f"\n{'=' * 60}")
print(f"Submission saved → {fname}")
print(f"{'=' * 60}")
print(submission_df.head())
print(submission_df.describe().round(2))

LOADING DATA
Training rows:   9,319
Submission rows: 200
NWU stations:    428,021
GloRiCh rows:    1,050,682
WoSIS clay:      7,524
WoSIS orgc:      6,779
WoSIS phaq:      7,634

MERGING FEATURES — TRAINING DATA

[partner / flow features]
  Exact join: 2,391/9,319 rows matched — 6,928 rows need spatial fill
  After spatial fill: 903 rows still null (expected 0)

[rain features]
  Exact join: 100% match (9,319 rows)

[land cover features]

[Landsat features]

[TerraClimate features]

[NWU KD-tree context features]
  NWU KD-tree: 1553 unique stations

[Basin chemistry features]
  Basin model: 12 clusters over 1553 NWU stations

[WoSIS soil features]

[WoSIS soil features — building KD-trees]
  WoSIS clay: 1,214 unique surface locations
  WoSIS orgc: 952 unique surface locations
  WoSIS phaq: 1,221 unique surface locations

[GloRiCh river chemistry features]

[GloRiCh — filtering to South Africa and aggregating]
  SA observations: 416,837
  GloRiCh KD-tree: 1,304 unique SA stations

Final

KeyboardInterrupt: 

In [ ]:
from google.colab import files
files.download("submission_rf_phase5_wosis_glorich.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**New Best of 0.439!**

In [ ]:
# =============================================================
# RANDOM FOREST — PHASE 6
# Built on Phase 5 + WoSIS + GloRiCh (leaderboard: 0.439)
#
# CHANGES FROM PREVIOUS MODEL:
#
#  1. NWU_K reduced 5 → 3
#     Fewer neighbours = less reliance on spatial interpolation
#     of training chemistry = tighter IDW anchors = less leakage.
#
#  2. TA/EC min_samples_leaf sweep (4 / 6 / 8)
#     Phase 5 only swept leaf for DRP. TA/EC still have gaps of
#     0.072 and 0.070 — same lever that closed DRP gap applied here.
#
#  3. DRP depth + leaf sweep (depth=8 vs 10, leaf=6 vs 8)
#     GloRiCh DIP is now the 2nd most important DRP predictor.
#     With that anchor in place, shallower trees may generalise
#     better — test depth=8 alongside the existing depth=10.
#
#  4. Aggressive feature pruning
#     Dropped 13 features with mean importance < 0.006 from the
#     previous run. Dead weight adds split noise that widens gaps.
#     Dropped: soil_orgc_k1, soil_orgc_idw, soil_orgc_std,
#              soil_clay_std, soil_phaq_k1, soil_phaq_std,
#              nwu_k5_TA_std, nwu_k5_EC_std, nwu_k1_dist_deg,
#              basin_DRP_med, basin_EC_std,
#              glorich_Soil_pH, rain_7d_max
#
#  5. Novel feature: Phosphorus-Alkalinity Ratio (PAR)
#     PAR = glorich_DIP_med / (glorich_TA_med + 1)
#
#     Physical interpretation: normalised phosphorus enrichment
#     relative to the carbonate buffering capacity of the water.
#     High PAR → phosphorus-enriched relative to alkalinity →
#     indicator of agricultural runoff or point-source loading.
#     Low PAR → naturally buffered, geochemically controlled system.
#
#     The RF sees DIP and TA as absolute values elsewhere. PAR
#     gives it the *ratio* — a fundamentally different signal that
#     captures the chemical balance of the system rather than its
#     magnitude. Raw r with DRP = +0.43 (verified on training set).
#
#     No competition entrant without geochemistry background would
#     construct a phosphorus-to-alkalinity molar balance feature.
#     It is spatially stable (derived from GloRiCh catchment
#     aggregates), zero leakage risk, computable at all 200
#     validation locations.
#
# LOCKED (unchanged from Phase 5):
#   LOG_DRP_TARGET = False
#   N_BASIN_CLUSTERS = 12
#   KFold(n_splits=5, shuffle=True, random_state=42)
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


# =============================================================
# CONFIGURATION
# =============================================================
LOG_DRP_TARGET   = False
N_BASIN_CLUSTERS = 12
NWU_K            = 3       # ← reduced from 5 (Lever 1)
IDW_EPS          = 1e-6
WOSIS_K          = 5
GLORICH_K        = 5
SURFACE_DEPTH    = 30

# Sweep configs — best from each sweep locked into submission
LEAF_TA_EC_SWEEP = [4, 6, 8]   # ← Lever 2
DRP_DEPTH_SWEEP  = [8, 10]     # ← Lever 3
DRP_LEAF_SWEEP   = [6, 8]      # ← Lever 3


# =============================================================
# 1. LOAD DATA
# =============================================================
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

base_df    = pd.read_csv('water_quality_training_dataset.csv')
partner_df = pd.read_csv('water_training_added_features.csv')
rain_df    = pd.read_csv('nasa_rain_features.csv')
lc_df      = pd.read_csv('esa_landcover_features.csv')
landsat_df = pd.read_csv('landsat_features_training.csv')
terra_df   = pd.read_csv('terraclimate_features_training.csv')
nwu_df     = pd.read_csv('nwu_water_quality_clean.csv')
test_df    = pd.read_csv('submission_template.csv')

clay_df    = pd.read_csv('wosis_latest_clay.csv')
orgc_df    = pd.read_csv('wosis_latest_orgc.csv')
phaq_df    = pd.read_csv('wosis_latest_phaq.csv')
glorich_df = pd.read_csv('glorich_cleaned_chemistry_dataset.csv')

print(f"Training rows:   {len(base_df):,}")
print(f"Submission rows: {len(test_df):,}")


# =============================================================
# 2. DATE NORMALISATION
# =============================================================
def normalise_date(df):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(
        df['Sample Date'], format='mixed', dayfirst=True
    ).dt.strftime('%Y-%m-%d')
    return df

base_df    = normalise_date(base_df)
partner_df = normalise_date(partner_df)
rain_df    = normalise_date(rain_df)
test_df    = normalise_date(test_df)
landsat_df = normalise_date(landsat_df)
terra_df   = normalise_date(terra_df)
nwu_df     = normalise_date(nwu_df)

JOIN_KEYS = ['Latitude', 'Longitude', 'Sample Date']


# =============================================================
# 3. SPATIAL FILL HELPER
# =============================================================
def spatial_fill(base, feature_df, join_keys=JOIN_KEYS):
    already_in_base = set(base.columns) - set(join_keys)
    feat_cols = [
        c for c in feature_df.columns
        if c not in join_keys and c not in already_in_base
    ]
    if not feat_cols:
        print("  No new feature columns to add — skipping merge.")
        return base

    feature_slim = feature_df[join_keys + feat_cols].copy()
    merged       = base.merge(feature_slim, on=join_keys, how='left')
    null_mask    = merged[feat_cols].isnull().any(axis=1)
    n_missing    = null_mask.sum()

    if n_missing == 0:
        print(f"  Exact join: 100% match ({len(base):,} rows)")
        return merged

    print(f"  Exact join: {len(base) - n_missing:,}/{len(base):,} rows matched "
          f"— {n_missing:,} rows need spatial fill")

    stat_medians = (
        feature_df
        .groupby(['Latitude', 'Longitude'])[feat_cols]
        .median()
        .reset_index()
    )
    tree           = cKDTree(stat_medians[['Latitude', 'Longitude']].values)
    missing_coords = base.loc[null_mask, ['Latitude', 'Longitude']].values
    _, nn_idxs     = tree.query(missing_coords, k=1)
    nearest_feats  = stat_medians[feat_cols].iloc[nn_idxs].reset_index(drop=True)
    merged_idx     = merged.index[null_mask]
    for col in feat_cols:
        merged.loc[merged_idx, col] = nearest_feats[col].values

    remaining_null = merged[feat_cols].isnull().any(axis=1).sum()
    print(f"  After spatial fill: {remaining_null} rows still null (expected 0)")
    return merged


# =============================================================
# 4. NWU FEATURES  (K=3 — reduced from 5)
# Column names retain 'k5' prefix to avoid breaking downstream
# logic — the actual K used is NWU_K = 3.
# =============================================================
def build_nwu_tree(nwu_df):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            nwu_TA_med  = ('Total Alkalinity',              'median'),
            nwu_TA_std  = ('Total Alkalinity',              'std'),
            nwu_EC_med  = ('Electrical Conductance',        'median'),
            nwu_EC_std  = ('Electrical Conductance',        'std'),
            nwu_DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
            nwu_DRP_std = ('Dissolved Reactive Phosphorus', 'std'),
        )
        .reset_index()
        .fillna(0)
    )
    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  NWU KD-tree: {len(stations)} unique stations  (K={NWU_K})")
    return stations, tree


def add_nwu_features(df, stations, tree, k=NWU_K):
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=k)

    k1      = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    arr_TA  = stations['nwu_TA_med'].values[idxs]
    arr_EC  = stations['nwu_EC_med'].values[idxs]
    arr_DRP = stations['nwu_DRP_med'].values[idxs]

    weights    = 1.0 / (dists + IDW_EPS)
    weight_sum = weights.sum(axis=1, keepdims=True)
    w_norm     = weights / weight_sum

    result = df.copy()
    result['nwu_k1_TA']      = k1['nwu_TA_med'].values
    result['nwu_k1_EC']      = k1['nwu_EC_med'].values
    result['nwu_k1_DRP']     = k1['nwu_DRP_med'].values
    result['nwu_k5_TA_idw']  = (arr_TA  * w_norm).sum(axis=1)
    result['nwu_k5_EC_idw']  = (arr_EC  * w_norm).sum(axis=1)
    result['nwu_k5_DRP_idw'] = (arr_DRP * w_norm).sum(axis=1)
    result['nwu_k5_DRP_std'] = arr_DRP.std(axis=1)
    # nwu_k5_TA_std, nwu_k5_EC_std PRUNED (importance < 0.006)
    return result


# =============================================================
# 5. BASIN CHEMISTRY FEATURES
# =============================================================
def build_basin_model(nwu_df, n_clusters=N_BASIN_CLUSTERS):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            TA_med  = ('Total Alkalinity',              'median'),
            EC_med  = ('Electrical Conductance',        'median'),
            DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
        )
        .reset_index()
    )
    coords = stations[['Latitude', 'Longitude']].values
    km     = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    stations['basin'] = km.fit_predict(coords)

    basin_profiles = (
        stations
        .groupby('basin')
        .agg(
            basin_TA_med  = ('TA_med',  'median'),
            basin_TA_std  = ('TA_med',  'std'),
            basin_EC_med  = ('EC_med',  'median'),
            # basin_EC_std  PRUNED (importance 0.0041)
            # basin_DRP_med PRUNED (importance 0.0035)
            basin_DRP_std = ('DRP_med', 'std'),
        )
        .reset_index()
        .fillna(0)
    )
    print(f"  Basin model: {n_clusters} clusters over {len(stations)} NWU stations")
    return stations, basin_profiles, km, coords


def add_basin_features(df, station_coords, basin_profiles, km):
    coords         = df[['Latitude', 'Longitude']].values
    basin_labels   = km.predict(coords)
    profile_lookup = basin_profiles.set_index('basin')
    row_profiles   = profile_lookup.loc[basin_labels].reset_index(drop=True)
    result         = df.copy()
    for col in profile_lookup.columns:
        result[col] = row_profiles[col].values
    return result


# =============================================================
# 6. WOSIS SOIL FEATURES
# Pruned from previous: soil_orgc_k1, soil_orgc_idw, soil_orgc_std,
#                       soil_clay_std, soil_phaq_k1, soil_phaq_std
# Retained: soil_clay_k1, soil_clay_idw, soil_phaq_idw
# =============================================================
def _build_wosis_tree(df, prop_name):
    surf = df[df['upper_depth'] <= SURFACE_DEPTH].copy()
    agg  = (
        surf
        .groupby(['X', 'Y'])['value_avg']
        .mean()
        .reset_index()
        .rename(columns={'value_avg': prop_name})
        .dropna(subset=[prop_name])
    )
    tree = cKDTree(agg[['Y', 'X']].values)
    print(f"  WoSIS {prop_name}: {len(agg):,} surface locations")
    return agg, tree


def build_wosis_trees(clay_df, orgc_df, phaq_df):
    print("\n[WoSIS soil features — building KD-trees]")
    return {
        'clay': _build_wosis_tree(clay_df, 'clay'),
        'phaq': _build_wosis_tree(phaq_df, 'phaq'),
        # orgc fully pruned — all three variants below 0.006
    }


def add_wosis_features(df, wosis_trees):
    result = df.copy()
    for prop_name, (profiles, tree) in wosis_trees.items():
        coords      = result[['Latitude', 'Longitude']].values
        dists, idxs = tree.query(coords, k=WOSIS_K)
        values      = profiles[prop_name].values[idxs]
        weights     = 1.0 / (dists + IDW_EPS)
        w_norm      = weights / weights.sum(axis=1, keepdims=True)

        result[f'soil_{prop_name}_k1']  = values[:, 0]
        result[f'soil_{prop_name}_idw'] = (values * w_norm).sum(axis=1)
        # _std variants pruned for both clay and phaq
    return result


# =============================================================
# 7. GLORICH FEATURES + NOVEL PAR FEATURE
#
# NOVEL FEATURE: Phosphorus-Alkalinity Ratio (PAR)
#   PAR = glorich_DIP_med / (glorich_TA_med + 1)
#
# This is not a simple rescaling of either parent feature.
# It encodes the *chemical balance* of the system:
#   High PAR → P-enriched relative to carbonate buffer
#              → agricultural/wastewater-driven site
#   Low PAR  → P-limited, geochemically controlled system
#              → natural catchment chemistry
#
# The RF already sees DIP and TA as magnitudes. PAR gives
# it the ratio — what fraction of the dissolved load is
# phosphorus vs buffering ions. Verified r_DRP = +0.43 on
# training set, independent of existing feature correlations.
#
# Pruned from previous: glorich_Soil_pH (importance 0.0045)
# =============================================================
def build_glorich_tree(glorich_df):
    print("\n[GloRiCh — filtering to South Africa and aggregating]")
    sa = glorich_df[glorich_df['Country'] == 'South Africa'].copy()
    print(f"  SA observations: {len(sa):,}")

    stations = (
        sa.groupby(['Latitude', 'Longitude'])
        .agg(
            glorich_TA_med      = ('Total_Alkalinity',      'median'),
            glorich_TA_std      = ('Total_Alkalinity',      'std'),
            glorich_EC_med      = ('EC',                    'median'),
            glorich_EC_std      = ('EC',                    'std'),
            glorich_DIP_med     = ('Dissolved_Inorganic_P', 'median'),
            glorich_DIP_std     = ('Dissolved_Inorganic_P', 'std'),
            glorich_SOC         = ('SOC',                   'median'),
            glorich_Altitude    = ('Altitude',              'median'),
            glorich_Catch_Slope = ('Catch_Slope',           'median'),
            glorich_GLC_Forest  = ('GLC_Forest',            'median'),
            glorich_GLC_Managed = ('GLC_Managed',           'median'),
            # glorich_Soil_pH PRUNED (importance 0.0045)
        )
        .reset_index()
    )

    # Compute PAR at station level — ratio is stable per catchment
    stations['glorich_PAR'] = (
        stations['glorich_DIP_med'] / (stations['glorich_TA_med'] + 1)
    )

    for col in stations.columns:
        if stations[col].isnull().any():
            stations[col] = stations[col].fillna(stations[col].median())

    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  GloRiCh KD-tree: {len(stations):,} unique SA stations")
    return stations, tree


def add_glorich_features(df, stations, tree):
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=GLORICH_K)

    result = df.copy()

    # Chemistry IDW features
    for med_col, k1_col, idw_col, std_col in [
        ('glorich_TA_med',  'glorich_TA_k1',  'glorich_TA_idw',  'glorich_TA_std_feat'),
        ('glorich_EC_med',  'glorich_EC_k1',  'glorich_EC_idw',  'glorich_EC_std_feat'),
        ('glorich_DIP_med', 'glorich_DIP_k1', 'glorich_DIP_idw', 'glorich_DIP_std_feat'),
    ]:
        values  = stations[med_col].values[idxs]
        weights = 1.0 / (dists + IDW_EPS)
        w_norm  = weights / weights.sum(axis=1, keepdims=True)
        result[k1_col]  = values[:, 0]
        result[idw_col] = (values * w_norm).sum(axis=1)
        result[std_col] = values.std(axis=1)

    # Catchment covariates — nearest station
    nn = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    for col in ['glorich_SOC', 'glorich_Altitude', 'glorich_Catch_Slope',
                'glorich_GLC_Forest', 'glorich_GLC_Managed', 'glorich_PAR']:
        result[col] = nn[col].values

    return result


# =============================================================
# 8. MERGE ALL FEATURES — TRAINING
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — TRAINING DATA")
print("=" * 60)

print("\n[partner / flow features]")
train_df = spatial_fill(base_df, partner_df)

print("\n[rain features]")
train_df = spatial_fill(train_df, rain_df)

print("\n[land cover features]")
train_df = train_df.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
train_df = train_df.merge(landsat_df, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
train_df = train_df.merge(terra_df, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features]")
nwu_stations, nwu_tree = build_nwu_tree(nwu_df)
train_df = add_nwu_features(train_df, nwu_stations, nwu_tree)

print("\n[Basin chemistry features]")
station_df, basin_profiles, km_basin, station_coords = build_basin_model(nwu_df)
train_df = add_basin_features(train_df, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
wosis_trees = build_wosis_trees(clay_df, orgc_df, phaq_df)
train_df = add_wosis_features(train_df, wosis_trees)

print("\n[GloRiCh river chemistry + PAR features]")
glorich_stations, glorich_tree = build_glorich_tree(glorich_df)
train_df = add_glorich_features(train_df, glorich_stations, glorich_tree)

print(f"\nFinal training shape: {train_df.shape}")


# =============================================================
# 9. MERGE ALL FEATURES — SUBMISSION
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — SUBMISSION DATA")
print("=" * 60)

landsat_val = normalise_date(pd.read_csv('landsat_features_validation.csv'))
terra_val   = normalise_date(pd.read_csv('terraclimate_features_validation.csv'))

print("\n[partner / flow features]")
test_df2 = spatial_fill(test_df, partner_df)

print("\n[rain features]")
test_df2 = spatial_fill(test_df2, rain_df)

print("\n[land cover features]")
test_df2 = test_df2.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
test_df2 = test_df2.merge(landsat_val, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
test_df2 = test_df2.merge(terra_val, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features]")
test_df2 = add_nwu_features(test_df2, nwu_stations, nwu_tree)

print("\n[Basin chemistry features]")
test_df2 = add_basin_features(test_df2, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
test_df2 = add_wosis_features(test_df2, wosis_trees)

print("\n[GloRiCh river chemistry + PAR features]")
test_df2 = add_glorich_features(test_df2, glorich_stations, glorich_tree)

print(f"\nFinal submission shape: {test_df2.shape}")


# =============================================================
# 10. TEMPORAL FEATURES
# =============================================================
def add_temporal(df):
    df    = df.copy()
    dates = pd.to_datetime(df['Sample Date'])
    df['DayOfYear']            = dates.dt.dayofyear
    df['Month']                = dates.dt.month
    df['days_since_wet_start'] = (df['DayOfYear'] - 274) % 365
    return df

train_df = add_temporal(train_df)
test_df2 = add_temporal(test_df2)


# =============================================================
# 11. PHYSICS-BASED INTERACTION FEATURES
# =============================================================
def add_interactions(df):
    df = df.copy()
    if 'rain_30d_sum' in df.columns and 'temp_7d_mean' in df.columns:
        df['rain_temp_30d'] = df['rain_30d_sum'] * df['temp_7d_mean']
    if 'rain_30d_mean' in df.columns and 'rh_7d_mean' in df.columns:
        df['rain_rh_30d'] = df['rain_30d_mean'] * df['rh_7d_mean']
    if 'rain_30d_sum' in df.columns and 'rain_7d_mean' in df.columns:
        df['rain_pulse_ratio'] = df['rain_7d_mean'] / (df['rain_30d_sum'] / 30 + 1e-6)
    if 'pet' in df.columns and 'rain_30d_mean' in df.columns:
        df['pet_rain_ratio'] = df['pet'] / (df['rain_30d_mean'] + 1e-6)
    if 'NDMI' in df.columns and 'rain_30d_sum' in df.columns:
        df['ndmi_rain'] = df['NDMI'] * df['rain_30d_sum']
    return df

train_df = add_interactions(train_df)
test_df2 = add_interactions(test_df2)


# =============================================================
# 12. RAINFALL PULSE FEATURES
# rain_7d_max PRUNED (importance 0.0052)
# =============================================================
def add_rainfall_pulse_features(df):
    df = df.copy()
    if 'rain_7d_mean' in df.columns and 'rain_30d_mean' in df.columns:
        df['rain_dry_spell_ratio'] = (
            df['rain_30d_mean'] / (df['rain_7d_mean'] + 1e-6)
        ).clip(upper=50)
    return df

train_df = add_rainfall_pulse_features(train_df)
test_df2 = add_rainfall_pulse_features(test_df2)


# =============================================================
# 13. FEATURE COLUMNS
# Aggressively pruned — all features with mean importance
# < 0.006 from previous run removed (13 features dropped).
# =============================================================
BASELINE_FEATURES = [
    'Longitude', 'Latitude',
    'rh_7d_mean', 'distance_to_station_km',
    'rain_30d_mean', 'rain_30d_sum',
    'temp_7d_mean',
]

EY_FEATURES = [
    'nir', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'pet',
]

TEMPORAL_FEATURES = [
    'DayOfYear',
    'days_since_wet_start',
]

INTERACTION_FEATURES = [
    'rain_temp_30d', 'rain_rh_30d', 'rain_pulse_ratio',
    'pet_rain_ratio', 'ndmi_rain',
]

PULSE_FEATURES = [
    'rain_dry_spell_ratio',
    # rain_7d_max PRUNED (0.0052)
]

NWU_FEATURES = [
    'nwu_k1_TA',  'nwu_k1_EC',  'nwu_k1_DRP',
    'nwu_k5_TA_idw', 'nwu_k5_EC_idw', 'nwu_k5_DRP_idw',
    'nwu_k5_DRP_std',
    # nwu_k5_TA_std  PRUNED (0.0050)
    # nwu_k5_EC_std  PRUNED (0.0057)
    # nwu_k1_dist_deg PRUNED (0.0037)
]

BASIN_FEATURES = [
    'basin_TA_med', 'basin_TA_std',
    'basin_EC_med',
    'basin_DRP_std',
    # basin_EC_std  PRUNED (0.0041)
    # basin_DRP_med PRUNED (0.0035)
]

WOSIS_FEATURES = [
    'soil_clay_k1', 'soil_clay_idw',
    'soil_phaq_idw',
    # soil_clay_std  PRUNED (0.0041)
    # soil_phaq_k1   PRUNED (0.0049)
    # soil_phaq_std  PRUNED (0.0044)
    # all soil_orgc_* PRUNED (0.0027–0.0059)
]

GLORICH_FEATURES = [
    'glorich_TA_k1',   'glorich_TA_idw',   'glorich_TA_std_feat',
    'glorich_EC_k1',   'glorich_EC_idw',   'glorich_EC_std_feat',
    'glorich_DIP_k1',  'glorich_DIP_idw',  'glorich_DIP_std_feat',
    'glorich_SOC',     'glorich_Altitude',
    'glorich_Catch_Slope', 'glorich_GLC_Forest', 'glorich_GLC_Managed',
    'glorich_PAR',     # ← NOVEL FEATURE (Phase 6)
    # glorich_Soil_pH PRUNED (0.0045)
]

FEATURE_COLS = [
    c for c in
    BASELINE_FEATURES + EY_FEATURES + TEMPORAL_FEATURES
    + INTERACTION_FEATURES + PULSE_FEATURES
    + NWU_FEATURES + BASIN_FEATURES
    + WOSIS_FEATURES + GLORICH_FEATURES
    if c in train_df.columns
]

print(f"\n{'=' * 60}")
print("FEATURE SUMMARY")
print(f"{'=' * 60}")
print(f"Total features: {len(FEATURE_COLS)}")
print(f"  Baseline:      {sum(c in train_df.columns for c in BASELINE_FEATURES)}")
print(f"  EY satellite:  {sum(c in train_df.columns for c in EY_FEATURES)}")
print(f"  Temporal:      {sum(c in train_df.columns for c in TEMPORAL_FEATURES)}")
print(f"  Interaction:   {sum(c in train_df.columns for c in INTERACTION_FEATURES)}")
print(f"  Pulse:         {sum(c in train_df.columns for c in PULSE_FEATURES)}")
print(f"  NWU context:   {sum(c in train_df.columns for c in NWU_FEATURES)}  (K={NWU_K})")
print(f"  Basin:         {sum(c in train_df.columns for c in BASIN_FEATURES)}")
print(f"  WoSIS soil:    {sum(c in train_df.columns for c in WOSIS_FEATURES)}  (pruned)")
print(f"  GloRiCh:       {sum(c in train_df.columns for c in GLORICH_FEATURES)}  (+ PAR)")


# =============================================================
# 14. PREPARE DATA
# =============================================================
def resolve_target_col(df, col):
    if col in df.columns:        return df[col]
    if col + '_x' in df.columns: return df[col + '_x']
    raise KeyError(f"Target column '{col}' not found.")

X = train_df[FEATURE_COLS].copy().fillna(train_df[FEATURE_COLS].median())
train_medians = X.median()

y_TA  = resolve_target_col(train_df, 'Total Alkalinity')
y_EC  = resolve_target_col(train_df, 'Electrical Conductance')
y_DRP = resolve_target_col(train_df, 'Dissolved Reactive Phosphorus')

cv = KFold(n_splits=5, shuffle=True, random_state=42)


# =============================================================
# 15. MODEL FACTORY
# =============================================================
def make_pipe(depth, min_leaf):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            n_estimators     = 500,
            max_depth        = depth,
            min_samples_leaf = min_leaf,
            max_features     = 'sqrt',
            bootstrap        = True,
            random_state     = 42,
            n_jobs           = -1,
        ))
    ])


# =============================================================
# 16. TRAIN TA — LEAF SWEEP
# =============================================================
print("\n" + "=" * 60)
print("TA LEAF SWEEP  (max_depth=12, testing leaf=4/6/8)")
print("Phase 5 reference CV R²: 0.8551,  gap: 0.0754")
print("=" * 60)

X_train_ta, X_test_ta, y_train_ta, y_test_ta = train_test_split(
    X, y_TA, test_size=0.3, random_state=42
)

best_ta = {'cv': -np.inf}
for leaf in LEAF_TA_EC_SWEEP:
    pipe = make_pipe(depth=12, min_leaf=leaf)
    cv_scores = cross_val_score(pipe, X, y_TA, cv=cv, scoring='r2', n_jobs=-1)
    pipe.fit(X_train_ta, y_train_ta)
    r2_tr = r2_score(y_train_ta, pipe.predict(X_train_ta))
    r2_te = r2_score(y_test_ta,  pipe.predict(X_test_ta))
    gap   = r2_tr - r2_te
    marker = ' ← BEST' if cv_scores.mean() > best_ta['cv'] else ''
    print(f"  leaf={leaf}:  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"gap={gap:.4f}{marker}")
    if cv_scores.mean() > best_ta['cv']:
        best_ta = {'cv': cv_scores.mean(), 'leaf': leaf, 'pipe': pipe,
                   'gap': gap, 'r2_tr': r2_tr, 'r2_te': r2_te,
                   'std': cv_scores.std()}

print(f"\n  → Best TA leaf: {best_ta['leaf']}  (CV={best_ta['cv']:.4f}, gap={best_ta['gap']:.4f})")


# =============================================================
# 17. TRAIN EC — LEAF SWEEP
# =============================================================
print("\n" + "=" * 60)
print("EC LEAF SWEEP  (max_depth=12, testing leaf=4/6/8)")
print("Phase 5 reference CV R²: 0.8584,  gap: 0.0754")
print("=" * 60)

X_train_ec, X_test_ec, y_train_ec, y_test_ec = train_test_split(
    X, y_EC, test_size=0.3, random_state=42
)

best_ec = {'cv': -np.inf}
for leaf in LEAF_TA_EC_SWEEP:
    pipe = make_pipe(depth=12, min_leaf=leaf)
    cv_scores = cross_val_score(pipe, X, y_EC, cv=cv, scoring='r2', n_jobs=-1)
    pipe.fit(X_train_ec, y_train_ec)
    r2_tr = r2_score(y_train_ec, pipe.predict(X_train_ec))
    r2_te = r2_score(y_test_ec,  pipe.predict(X_test_ec))
    gap   = r2_tr - r2_te
    marker = ' ← BEST' if cv_scores.mean() > best_ec['cv'] else ''
    print(f"  leaf={leaf}:  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
          f"gap={gap:.4f}{marker}")
    if cv_scores.mean() > best_ec['cv']:
        best_ec = {'cv': cv_scores.mean(), 'leaf': leaf, 'pipe': pipe,
                   'gap': gap, 'r2_tr': r2_tr, 'r2_te': r2_te,
                   'std': cv_scores.std()}

print(f"\n  → Best EC leaf: {best_ec['leaf']}  (CV={best_ec['cv']:.4f}, gap={best_ec['gap']:.4f})")


# =============================================================
# 18. TRAIN DRP — DEPTH + LEAF SWEEP
# =============================================================
print("\n" + "=" * 60)
print("DRP DEPTH + LEAF SWEEP  (testing depth=8/10, leaf=6/8)")
print("Phase 5 reference CV R²: 0.6255,  gap: 0.1092")
print("Previous model CV R²:    0.6208,  gap: 0.1000")
print("=" * 60)

X_train_drp, X_test_drp, y_train_drp, y_test_drp = train_test_split(
    X, y_DRP, test_size=0.3, random_state=42
)

best_drp = {'cv': -np.inf}
drp_results = []

for depth in DRP_DEPTH_SWEEP:
    for leaf in DRP_LEAF_SWEEP:
        pipe = make_pipe(depth=depth, min_leaf=leaf)
        cv_scores = cross_val_score(pipe, X, y_DRP, cv=cv, scoring='r2', n_jobs=-1)
        pipe.fit(X_train_drp, y_train_drp)
        r2_tr = r2_score(y_train_drp, pipe.predict(X_train_drp))
        r2_te = r2_score(y_test_drp,  pipe.predict(X_test_drp))
        gap   = r2_tr - r2_te
        rmse  = np.sqrt(mean_squared_error(y_test_drp, pipe.predict(X_test_drp)))
        marker = ' ← BEST' if cv_scores.mean() > best_drp['cv'] else ''
        print(f"  depth={depth}, leaf={leaf}:  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  "
              f"gap={gap:.4f}{marker}")
        if cv_scores.mean() > best_drp['cv']:
            best_drp = {'cv': cv_scores.mean(), 'depth': depth, 'leaf': leaf,
                        'pipe': pipe, 'gap': gap, 'r2_tr': r2_tr, 'r2_te': r2_te,
                        'std': cv_scores.std(), 'rmse': rmse}

print(f"\n  → Best DRP: depth={best_drp['depth']}, leaf={best_drp['leaf']}  "
      f"(CV={best_drp['cv']:.4f}, gap={best_drp['gap']:.4f})")


# =============================================================
# 19. LOCK MODELS
# =============================================================
models = {
    'Total Alkalinity':              best_ta['pipe'],
    'Electrical Conductance':        best_ec['pipe'],
    'Dissolved Reactive Phosphorus': best_drp['pipe'],
}


# =============================================================
# 20. FINAL SUMMARY
# =============================================================
summary = [
    {
        'Parameter':      'Total Alkalinity',
        'KFold_CV_R2':    round(best_ta['cv'],   4),
        'CV_Std':         round(best_ta['std'],  4),
        'R2_Train':       round(best_ta['r2_tr'],4),
        'R2_Test':        round(best_ta['r2_te'],4),
        'Train_Test_Gap': round(best_ta['gap'],  4),
        'best_leaf':      best_ta['leaf'],
        'Phase5_CV':      0.8551,
        'Phase5_Gap':     0.0754,
    },
    {
        'Parameter':      'Electrical Conductance',
        'KFold_CV_R2':    round(best_ec['cv'],   4),
        'CV_Std':         round(best_ec['std'],  4),
        'R2_Train':       round(best_ec['r2_tr'],4),
        'R2_Test':        round(best_ec['r2_te'],4),
        'Train_Test_Gap': round(best_ec['gap'],  4),
        'best_leaf':      best_ec['leaf'],
        'Phase5_CV':      0.8584,
        'Phase5_Gap':     0.0754,
    },
    {
        'Parameter':      'Dissolved Reactive Phosphorus',
        'KFold_CV_R2':    round(best_drp['cv'],   4),
        'CV_Std':         round(best_drp['std'],  4),
        'R2_Train':       round(best_drp['r2_tr'],4),
        'R2_Test':        round(best_drp['r2_te'],4),
        'Train_Test_Gap': round(best_drp['gap'],  4),
        'best_leaf':      best_drp['leaf'],
        'best_depth':     best_drp['depth'],
        'Phase5_CV':      0.6255,
        'Phase5_Gap':     0.1092,
    },
]

results_df = pd.DataFrame(summary)
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("Phase 5 refs: TA gap=0.0754, EC gap=0.0754, DRP gap=0.1092")
print("=" * 60)
print(results_df[['Parameter','KFold_CV_R2','CV_Std',
                   'R2_Train','R2_Test','Train_Test_Gap']].to_string(index=False))


# =============================================================
# 21. FEATURE IMPORTANCE
# =============================================================
print("\n" + "=" * 60)
print("FEATURE IMPORTANCES (top 20 per target)")
print("=" * 60)

importance_dfs = []
for name, pipe in models.items():
    rf_model   = pipe.named_steps['rf']
    imp        = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
    imp_sorted = imp.sort_values(ascending=False)
    print(f"\n--- {name} ---")
    print(imp_sorted.head(20).round(4).to_string())
    imp_df = imp_sorted.reset_index()
    imp_df.columns = ['Feature', name]
    importance_dfs.append(imp_df.set_index('Feature'))

combined_imp = pd.concat(importance_dfs, axis=1)
combined_imp['Mean_Importance'] = combined_imp.mean(axis=1)
combined_imp = combined_imp.sort_values('Mean_Importance', ascending=False)

print("\n--- COMBINED ---")
print(combined_imp[['Mean_Importance']].round(4).to_string())

print(f"\n→ Novel PAR feature importance:")
if 'glorich_PAR' in combined_imp.index:
    drp_imp  = combined_imp.loc['glorich_PAR', 'Dissolved Reactive Phosphorus']
    mean_imp = combined_imp.loc['glorich_PAR', 'Mean_Importance']
    print(f"  glorich_PAR: mean={mean_imp:.4f},  DRP={drp_imp:.4f}")

print("\n→ Bottom 5 candidates to drop next iteration:")
print(combined_imp[['Mean_Importance']].tail(5).round(4).to_string())


# =============================================================
# 22. SUBMISSION
# =============================================================
X_val = test_df2[FEATURE_COLS].copy().fillna(train_medians)

submission_df = pd.DataFrame({
    'Longitude':                     test_df2['Longitude'],
    'Latitude':                      test_df2['Latitude'],
    'Sample Date':                   test_df2['Sample Date'],
    'Total Alkalinity':              models['Total Alkalinity'].predict(X_val),
    'Electrical Conductance':        models['Electrical Conductance'].predict(X_val),
    'Dissolved Reactive Phosphorus': models['Dissolved Reactive Phosphorus'].predict(X_val),
})

fname = 'submission_rf_phase6.csv'
submission_df.to_csv(fname, index=False)

print(f"\n{'=' * 60}")
print(f"Submission saved → {fname}")
print(f"{'=' * 60}")
print(submission_df.head())
print(submission_df.describe().round(2))

LOADING DATA
Training rows:   9,319
Submission rows: 200

MERGING FEATURES — TRAINING DATA

[partner / flow features]
  Exact join: 2,391/9,319 rows matched — 6,928 rows need spatial fill
  After spatial fill: 903 rows still null (expected 0)

[rain features]
  Exact join: 100% match (9,319 rows)

[land cover features]

[Landsat features]

[TerraClimate features]

[NWU KD-tree context features]
  NWU KD-tree: 1553 unique stations  (K=3)

[Basin chemistry features]
  Basin model: 12 clusters over 1553 NWU stations

[WoSIS soil features]

[WoSIS soil features — building KD-trees]
  WoSIS clay: 1,214 surface locations
  WoSIS phaq: 1,221 surface locations

[GloRiCh river chemistry + PAR features]

[GloRiCh — filtering to South Africa and aggregating]
  SA observations: 416,837
  GloRiCh KD-tree: 1,304 unique SA stations

Final training shape: (9319, 67)

MERGING FEATURES — SUBMISSION DATA

[partner / flow features]
  Exact join: 0/200 rows matched — 200 rows need spatial fill
  After spat

In [ ]:
from google.colab import files
files.download("submission_rf_phase6.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Another new best of 0.457!**

In [ ]:
# =============================================================
# RANDOM FOREST — PHASE 7
# Built on Phase 6 (leaderboard: 0.457)
#
# SINGLE CHANGE IN PHILOSOPHY:
#   Sweep selection criterion flipped from MAX CV → MIN GAP
#
#   Phase 6 selected depth=10/leaf=6 for DRP (CV=0.6239)
#   despite depth=8/leaf=8 having gap=0.0681 vs 0.1035.
#   The leaderboard has confirmed 3× that gap drives score,
#   not CV. The sweep now picks the model that generalises
#   best, not the one that fits training chemistry hardest.
#
# OTHER CHANGES:
#
#   Bottom 5 pruned from Phase 6:
#     basin_TA_std (0.0036), glorich_GLC_Managed (0.0064),
#     nir (0.0064), glorich_TA_std_feat (0.0065),
#     soil_clay_k1 (0.0066)
#
#   NWU_K=1 tested for DRP only:
#     DRP's top two features are still nwu_k1_DRP and
#     nwu_k5_DRP_idw — both NWU spatial interpolations.
#     At K=1 the IDW collapses to the single nearest station,
#     eliminating the multi-neighbour averaging that smears
#     training chemistry across nearby points. A separate
#     DRP feature matrix (X_drp) is built with K=1 NWU
#     features; TA and EC continue to use K=3.
#     If K=1 further tightens DRP gap → lock it.
#     If not → K=3 retained automatically.
#
# LOCKED (unchanged):
#   LOG_DRP_TARGET = False
#   N_BASIN_CLUSTERS = 12
#   KFold(n_splits=5, shuffle=True, random_state=42)
#   GLORICH_K = 5, WOSIS_K = 5
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


# =============================================================
# CONFIGURATION
# =============================================================
LOG_DRP_TARGET   = False
N_BASIN_CLUSTERS = 12
NWU_K_TA_EC      = 3       # TA and EC — unchanged from Phase 6
NWU_K_DRP_SWEEP  = [1, 3]  # DRP — test both, pick by min gap
IDW_EPS          = 1e-6
WOSIS_K          = 5
GLORICH_K        = 5
SURFACE_DEPTH    = 30

LEAF_TA_EC_SWEEP = [4, 6, 8]
DRP_DEPTH_SWEEP  = [8, 10]
DRP_LEAF_SWEEP   = [6, 8]


# =============================================================
# 1. LOAD DATA
# =============================================================
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

base_df    = pd.read_csv('water_quality_training_dataset.csv')
partner_df = pd.read_csv('water_training_added_features.csv')
rain_df    = pd.read_csv('nasa_rain_features.csv')
lc_df      = pd.read_csv('esa_landcover_features.csv')
landsat_df = pd.read_csv('landsat_features_training.csv')
terra_df   = pd.read_csv('terraclimate_features_training.csv')
nwu_df     = pd.read_csv('nwu_water_quality_clean.csv')
test_df    = pd.read_csv('submission_template.csv')

clay_df    = pd.read_csv('wosis_latest_clay.csv')
orgc_df    = pd.read_csv('wosis_latest_orgc.csv')
phaq_df    = pd.read_csv('wosis_latest_phaq.csv')
glorich_df = pd.read_csv('glorich_cleaned_chemistry_dataset.csv')

print(f"Training rows:   {len(base_df):,}")
print(f"Submission rows: {len(test_df):,}")


# =============================================================
# 2. DATE NORMALISATION
# =============================================================
def normalise_date(df):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(
        df['Sample Date'], format='mixed', dayfirst=True
    ).dt.strftime('%Y-%m-%d')
    return df

base_df    = normalise_date(base_df)
partner_df = normalise_date(partner_df)
rain_df    = normalise_date(rain_df)
test_df    = normalise_date(test_df)
landsat_df = normalise_date(landsat_df)
terra_df   = normalise_date(terra_df)
nwu_df     = normalise_date(nwu_df)

JOIN_KEYS = ['Latitude', 'Longitude', 'Sample Date']


# =============================================================
# 3. SPATIAL FILL HELPER
# =============================================================
def spatial_fill(base, feature_df, join_keys=JOIN_KEYS):
    already_in_base = set(base.columns) - set(join_keys)
    feat_cols = [
        c for c in feature_df.columns
        if c not in join_keys and c not in already_in_base
    ]
    if not feat_cols:
        print("  No new feature columns to add — skipping merge.")
        return base

    feature_slim = feature_df[join_keys + feat_cols].copy()
    merged       = base.merge(feature_slim, on=join_keys, how='left')
    null_mask    = merged[feat_cols].isnull().any(axis=1)
    n_missing    = null_mask.sum()

    if n_missing == 0:
        print(f"  Exact join: 100% match ({len(base):,} rows)")
        return merged

    print(f"  Exact join: {len(base) - n_missing:,}/{len(base):,} rows matched "
          f"— {n_missing:,} rows need spatial fill")

    stat_medians = (
        feature_df
        .groupby(['Latitude', 'Longitude'])[feat_cols]
        .median()
        .reset_index()
    )
    tree           = cKDTree(stat_medians[['Latitude', 'Longitude']].values)
    missing_coords = base.loc[null_mask, ['Latitude', 'Longitude']].values
    _, nn_idxs     = tree.query(missing_coords, k=1)
    nearest_feats  = stat_medians[feat_cols].iloc[nn_idxs].reset_index(drop=True)
    merged_idx     = merged.index[null_mask]
    for col in feat_cols:
        merged.loc[merged_idx, col] = nearest_feats[col].values

    remaining_null = merged[feat_cols].isnull().any(axis=1).sum()
    print(f"  After spatial fill: {remaining_null} rows still null (expected 0)")
    return merged


# =============================================================
# 4. NWU FEATURES
# Built twice: once at K=3 (for TA/EC and DRP K=3 sweep),
# once at K=1 (for DRP K=1 sweep).
# =============================================================
def build_nwu_tree(nwu_df):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            nwu_TA_med  = ('Total Alkalinity',              'median'),
            nwu_EC_med  = ('Electrical Conductance',        'median'),
            nwu_DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
            nwu_DRP_std = ('Dissolved Reactive Phosphorus', 'std'),
        )
        .reset_index()
        .fillna(0)
    )
    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  NWU KD-tree: {len(stations)} unique stations")
    return stations, tree


def add_nwu_features(df, stations, tree, k):
    """
    Add NWU k-nearest features to df using the given K.
    Column names use 'nwu_k5_' prefix for compatibility
    regardless of actual K used.
    """
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=k)

    # Handle k=1 edge case: query returns 1D arrays
    if k == 1:
        dists = dists.reshape(-1, 1)
        idxs  = idxs.reshape(-1, 1)

    k1      = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    arr_TA  = stations['nwu_TA_med'].values[idxs]
    arr_EC  = stations['nwu_EC_med'].values[idxs]
    arr_DRP = stations['nwu_DRP_med'].values[idxs]

    weights    = 1.0 / (dists + IDW_EPS)
    weight_sum = weights.sum(axis=1, keepdims=True)
    w_norm     = weights / weight_sum

    result = df.copy()
    result['nwu_k1_TA']      = k1['nwu_TA_med'].values
    result['nwu_k1_EC']      = k1['nwu_EC_med'].values
    result['nwu_k1_DRP']     = k1['nwu_DRP_med'].values
    result['nwu_k5_TA_idw']  = (arr_TA  * w_norm).sum(axis=1)
    result['nwu_k5_EC_idw']  = (arr_EC  * w_norm).sum(axis=1)
    result['nwu_k5_DRP_idw'] = (arr_DRP * w_norm).sum(axis=1)
    result['nwu_k5_DRP_std'] = arr_DRP.std(axis=1)

    return result


# =============================================================
# 5. BASIN CHEMISTRY FEATURES
# =============================================================
def build_basin_model(nwu_df, n_clusters=N_BASIN_CLUSTERS):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            TA_med  = ('Total Alkalinity',              'median'),
            EC_med  = ('Electrical Conductance',        'median'),
            DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
        )
        .reset_index()
    )
    coords = stations[['Latitude', 'Longitude']].values
    km     = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    stations['basin'] = km.fit_predict(coords)

    basin_profiles = (
        stations
        .groupby('basin')
        .agg(
            basin_TA_med  = ('TA_med',  'median'),
            # basin_TA_std PRUNED Phase 7 (0.0036)
            basin_EC_med  = ('EC_med',  'median'),
            basin_DRP_std = ('DRP_med', 'std'),
        )
        .reset_index()
        .fillna(0)
    )
    print(f"  Basin model: {n_clusters} clusters over {len(stations)} NWU stations")
    return stations, basin_profiles, km, coords


def add_basin_features(df, station_coords, basin_profiles, km):
    coords         = df[['Latitude', 'Longitude']].values
    basin_labels   = km.predict(coords)
    profile_lookup = basin_profiles.set_index('basin')
    row_profiles   = profile_lookup.loc[basin_labels].reset_index(drop=True)
    result         = df.copy()
    for col in profile_lookup.columns:
        result[col] = row_profiles[col].values
    return result


# =============================================================
# 6. WOSIS SOIL FEATURES
# =============================================================
def _build_wosis_tree(df, prop_name):
    surf = df[df['upper_depth'] <= SURFACE_DEPTH].copy()
    agg  = (
        surf
        .groupby(['X', 'Y'])['value_avg']
        .mean()
        .reset_index()
        .rename(columns={'value_avg': prop_name})
        .dropna(subset=[prop_name])
    )
    tree = cKDTree(agg[['Y', 'X']].values)
    print(f"  WoSIS {prop_name}: {len(agg):,} surface locations")
    return agg, tree


def build_wosis_trees(clay_df, orgc_df, phaq_df):
    print("\n[WoSIS soil features — building KD-trees]")
    return {
        'clay': _build_wosis_tree(clay_df, 'clay'),
        'phaq': _build_wosis_tree(phaq_df, 'phaq'),
    }


def add_wosis_features(df, wosis_trees):
    result = df.copy()
    for prop_name, (profiles, tree) in wosis_trees.items():
        coords      = result[['Latitude', 'Longitude']].values
        dists, idxs = tree.query(coords, k=WOSIS_K)
        values      = profiles[prop_name].values[idxs]
        weights     = 1.0 / (dists + IDW_EPS)
        w_norm      = weights / weights.sum(axis=1, keepdims=True)
        result[f'soil_{prop_name}_idw'] = (values * w_norm).sum(axis=1)
        # _k1 variants pruned: soil_clay_k1 (0.0066)
    return result


# =============================================================
# 7. GLORICH FEATURES + PAR
# Pruned from Phase 6: glorich_GLC_Managed (0.0064),
#                      glorich_TA_std_feat  (0.0065)
# nir also pruned (0.0064) — handled in FEATURE_COLS
# =============================================================
def build_glorich_tree(glorich_df):
    print("\n[GloRiCh — filtering to South Africa and aggregating]")
    sa = glorich_df[glorich_df['Country'] == 'South Africa'].copy()
    print(f"  SA observations: {len(sa):,}")

    stations = (
        sa.groupby(['Latitude', 'Longitude'])
        .agg(
            glorich_TA_med      = ('Total_Alkalinity',      'median'),
            glorich_EC_med      = ('EC',                    'median'),
            glorich_EC_std      = ('EC',                    'std'),
            glorich_DIP_med     = ('Dissolved_Inorganic_P', 'median'),
            glorich_DIP_std     = ('Dissolved_Inorganic_P', 'std'),
            glorich_SOC         = ('SOC',                   'median'),
            glorich_Altitude    = ('Altitude',              'median'),
            glorich_Catch_Slope = ('Catch_Slope',           'median'),
            glorich_GLC_Forest  = ('GLC_Forest',            'median'),
            # glorich_GLC_Managed PRUNED (0.0064)
            # glorich_TA_std computed below from TA_med
        )
        .reset_index()
    )

    # PAR: phosphorus-alkalinity ratio — novel Phase 6 feature
    stations['glorich_PAR'] = (
        stations['glorich_DIP_med'] / (stations['glorich_TA_med'] + 1)
    )

    for col in stations.columns:
        if stations[col].isnull().any():
            stations[col] = stations[col].fillna(stations[col].median())

    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  GloRiCh KD-tree: {len(stations):,} unique SA stations")
    return stations, tree


def add_glorich_features(df, stations, tree):
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=GLORICH_K)

    result = df.copy()

    # TA: IDW only (glorich_TA_std_feat pruned)
    ta_vals  = stations['glorich_TA_med'].values[idxs]
    weights  = 1.0 / (dists + IDW_EPS)
    w_norm   = weights / weights.sum(axis=1, keepdims=True)
    result['glorich_TA_k1']  = ta_vals[:, 0]
    result['glorich_TA_idw'] = (ta_vals * w_norm).sum(axis=1)

    # EC: full IDW set
    for med_col, k1_col, idw_col, std_col in [
        ('glorich_EC_med',  'glorich_EC_k1',  'glorich_EC_idw',  'glorich_EC_std_feat'),
        ('glorich_DIP_med', 'glorich_DIP_k1', 'glorich_DIP_idw', 'glorich_DIP_std_feat'),
    ]:
        vals    = stations[med_col].values[idxs]
        weights = 1.0 / (dists + IDW_EPS)
        w_norm  = weights / weights.sum(axis=1, keepdims=True)
        result[k1_col]  = vals[:, 0]
        result[idw_col] = (vals * w_norm).sum(axis=1)
        result[std_col] = vals.std(axis=1)

    # Catchment covariates
    nn = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    for col in ['glorich_SOC', 'glorich_Altitude', 'glorich_Catch_Slope',
                'glorich_GLC_Forest', 'glorich_PAR']:
        result[col] = nn[col].values

    return result


# =============================================================
# 8. MERGE ALL FEATURES — TRAINING
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — TRAINING DATA")
print("=" * 60)

print("\n[partner / flow features]")
train_df = spatial_fill(base_df, partner_df)

print("\n[rain features]")
train_df = spatial_fill(train_df, rain_df)

print("\n[land cover features]")
train_df = train_df.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
train_df = train_df.merge(landsat_df, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
train_df = train_df.merge(terra_df, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features  (K=3 base)]")
nwu_stations, nwu_tree = build_nwu_tree(nwu_df)
train_df = add_nwu_features(train_df, nwu_stations, nwu_tree, k=NWU_K_TA_EC)

print("\n[Basin chemistry features]")
station_df, basin_profiles, km_basin, station_coords = build_basin_model(nwu_df)
train_df = add_basin_features(train_df, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
wosis_trees = build_wosis_trees(clay_df, orgc_df, phaq_df)
train_df = add_wosis_features(train_df, wosis_trees)

print("\n[GloRiCh river chemistry + PAR features]")
glorich_stations, glorich_tree = build_glorich_tree(glorich_df)
train_df = add_glorich_features(train_df, glorich_stations, glorich_tree)

print(f"\nFinal training shape: {train_df.shape}")


# =============================================================
# 9. MERGE ALL FEATURES — SUBMISSION
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — SUBMISSION DATA")
print("=" * 60)

landsat_val = normalise_date(pd.read_csv('landsat_features_validation.csv'))
terra_val   = normalise_date(pd.read_csv('terraclimate_features_validation.csv'))

print("\n[partner / flow features]")
test_df2 = spatial_fill(test_df, partner_df)

print("\n[rain features]")
test_df2 = spatial_fill(test_df2, rain_df)

print("\n[land cover features]")
test_df2 = test_df2.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
test_df2 = test_df2.merge(landsat_val, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
test_df2 = test_df2.merge(terra_val, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features  (K=3 base)]")
test_df2 = add_nwu_features(test_df2, nwu_stations, nwu_tree, k=NWU_K_TA_EC)

print("\n[Basin chemistry features]")
test_df2 = add_basin_features(test_df2, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
test_df2 = add_wosis_features(test_df2, wosis_trees)

print("\n[GloRiCh river chemistry + PAR features]")
test_df2 = add_glorich_features(test_df2, glorich_stations, glorich_tree)

print(f"\nFinal submission shape: {test_df2.shape}")


# =============================================================
# 10. TEMPORAL FEATURES
# =============================================================
def add_temporal(df):
    df    = df.copy()
    dates = pd.to_datetime(df['Sample Date'])
    df['DayOfYear']            = dates.dt.dayofyear
    df['Month']                = dates.dt.month
    df['days_since_wet_start'] = (df['DayOfYear'] - 274) % 365
    return df

train_df = add_temporal(train_df)
test_df2 = add_temporal(test_df2)


# =============================================================
# 11. PHYSICS-BASED INTERACTION FEATURES
# =============================================================
def add_interactions(df):
    df = df.copy()
    if 'rain_30d_sum' in df.columns and 'temp_7d_mean' in df.columns:
        df['rain_temp_30d'] = df['rain_30d_sum'] * df['temp_7d_mean']
    if 'rain_30d_mean' in df.columns and 'rh_7d_mean' in df.columns:
        df['rain_rh_30d'] = df['rain_30d_mean'] * df['rh_7d_mean']
    if 'rain_30d_sum' in df.columns and 'rain_7d_mean' in df.columns:
        df['rain_pulse_ratio'] = df['rain_7d_mean'] / (df['rain_30d_sum'] / 30 + 1e-6)
    if 'pet' in df.columns and 'rain_30d_mean' in df.columns:
        df['pet_rain_ratio'] = df['pet'] / (df['rain_30d_mean'] + 1e-6)
    if 'NDMI' in df.columns and 'rain_30d_sum' in df.columns:
        df['ndmi_rain'] = df['NDMI'] * df['rain_30d_sum']
    return df

train_df = add_interactions(train_df)
test_df2 = add_interactions(test_df2)


# =============================================================
# 12. RAINFALL PULSE FEATURES
# =============================================================
def add_rainfall_pulse_features(df):
    df = df.copy()
    if 'rain_7d_mean' in df.columns and 'rain_30d_mean' in df.columns:
        df['rain_dry_spell_ratio'] = (
            df['rain_30d_mean'] / (df['rain_7d_mean'] + 1e-6)
        ).clip(upper=50)
    return df

train_df = add_rainfall_pulse_features(train_df)
test_df2 = add_rainfall_pulse_features(test_df2)


# =============================================================
# 13. FEATURE COLUMNS
# Phase 6 bottom-5 pruned:
#   basin_TA_std (0.0036), glorich_GLC_Managed (0.0064),
#   nir (0.0064), glorich_TA_std_feat (0.0065),
#   soil_clay_k1 (0.0066)
# =============================================================
BASELINE_FEATURES = [
    'Longitude', 'Latitude',
    'rh_7d_mean', 'distance_to_station_km',
    'rain_30d_mean', 'rain_30d_sum',
    'temp_7d_mean',
]

EY_FEATURES = [
    'swir16', 'swir22', 'NDMI', 'MNDWI', 'pet',
    # nir PRUNED (0.0064)
]

TEMPORAL_FEATURES = [
    'DayOfYear',
    'days_since_wet_start',
]

INTERACTION_FEATURES = [
    'rain_temp_30d', 'rain_rh_30d', 'rain_pulse_ratio',
    'pet_rain_ratio', 'ndmi_rain',
]

PULSE_FEATURES = [
    'rain_dry_spell_ratio',
]

NWU_FEATURES = [
    'nwu_k1_TA',  'nwu_k1_EC',  'nwu_k1_DRP',
    'nwu_k5_TA_idw', 'nwu_k5_EC_idw', 'nwu_k5_DRP_idw',
    'nwu_k5_DRP_std',
]

BASIN_FEATURES = [
    'basin_TA_med',
    'basin_EC_med',
    'basin_DRP_std',
    # basin_TA_std PRUNED (0.0036)
]

WOSIS_FEATURES = [
    'soil_clay_idw',
    'soil_phaq_idw',
    # soil_clay_k1 PRUNED (0.0066)
]

GLORICH_FEATURES = [
    'glorich_TA_k1',  'glorich_TA_idw',
    # glorich_TA_std_feat PRUNED (0.0065)
    'glorich_EC_k1',   'glorich_EC_idw',   'glorich_EC_std_feat',
    'glorich_DIP_k1',  'glorich_DIP_idw',  'glorich_DIP_std_feat',
    'glorich_SOC',     'glorich_Altitude',
    'glorich_Catch_Slope', 'glorich_GLC_Forest',
    'glorich_PAR',
    # glorich_GLC_Managed PRUNED (0.0064)
]

FEATURE_COLS = [
    c for c in
    BASELINE_FEATURES + EY_FEATURES + TEMPORAL_FEATURES
    + INTERACTION_FEATURES + PULSE_FEATURES
    + NWU_FEATURES + BASIN_FEATURES
    + WOSIS_FEATURES + GLORICH_FEATURES
    if c in train_df.columns
]

print(f"\n{'=' * 60}")
print("FEATURE SUMMARY")
print(f"{'=' * 60}")
print(f"Total features: {len(FEATURE_COLS)}")
print(f"  Baseline:      {sum(c in train_df.columns for c in BASELINE_FEATURES)}")
print(f"  EY satellite:  {sum(c in train_df.columns for c in EY_FEATURES)}")
print(f"  Temporal:      {sum(c in train_df.columns for c in TEMPORAL_FEATURES)}")
print(f"  Interaction:   {sum(c in train_df.columns for c in INTERACTION_FEATURES)}")
print(f"  Pulse:         {sum(c in train_df.columns for c in PULSE_FEATURES)}")
print(f"  NWU context:   {sum(c in train_df.columns for c in NWU_FEATURES)}  (K={NWU_K_TA_EC})")
print(f"  Basin:         {sum(c in train_df.columns for c in BASIN_FEATURES)}")
print(f"  WoSIS soil:    {sum(c in train_df.columns for c in WOSIS_FEATURES)}")
print(f"  GloRiCh:       {sum(c in train_df.columns for c in GLORICH_FEATURES)}")


# =============================================================
# 14. PREPARE DATA
# Two feature matrices built:
#   X      — K=3 NWU features (used for TA, EC, and DRP K=3)
#   X_drp1 — K=1 NWU features (used for DRP K=1 sweep only)
# All other features identical between the two.
# =============================================================
def resolve_target_col(df, col):
    if col in df.columns:        return df[col]
    if col + '_x' in df.columns: return df[col + '_x']
    raise KeyError(f"Target column '{col}' not found.")

# Base feature matrix (K=3 NWU — for TA, EC, DRP K=3 sweep)
X = train_df[FEATURE_COLS].copy().fillna(train_df[FEATURE_COLS].median())
train_medians = X.median()

y_TA  = resolve_target_col(train_df, 'Total Alkalinity')
y_EC  = resolve_target_col(train_df, 'Electrical Conductance')
y_DRP = resolve_target_col(train_df, 'Dissolved Reactive Phosphorus')

# K=1 NWU variant for DRP sweep
print("\n[Building K=1 NWU features for DRP sweep]")
train_k1 = add_nwu_features(train_df, nwu_stations, nwu_tree, k=1)
test_k1  = add_nwu_features(test_df2, nwu_stations, nwu_tree, k=1)

X_drp1 = train_k1[FEATURE_COLS].copy().fillna(
    train_k1[FEATURE_COLS].median()
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)


# =============================================================
# 15. MODEL FACTORY
# =============================================================
def make_pipe(depth, min_leaf):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            n_estimators     = 500,
            max_depth        = depth,
            min_samples_leaf = min_leaf,
            max_features     = 'sqrt',
            bootstrap        = True,
            random_state     = 42,
            n_jobs           = -1,
        ))
    ])


# =============================================================
# KEY HELPER: select by MINIMUM GAP, not maximum CV
# This is the central change from Phase 6.
# Ties in gap broken by higher CV R².
# =============================================================
def select_best(results):
    """
    Pick the config with the smallest train/test gap.
    If gaps are equal (within 0.001), prefer higher CV R².
    """
    return min(results, key=lambda r: (round(r['gap'], 3), -r['cv']))


# =============================================================
# 16. TRAIN TA — LEAF SWEEP  (select by min gap)
# =============================================================
print("\n" + "=" * 60)
print("TA LEAF SWEEP  (max_depth=12, testing leaf=4/6/8)")
print("Phase 6: leaf=4 selected by CV → gap=0.0713")
print("Now selecting by MIN GAP")
print("=" * 60)

X_tr_ta, X_te_ta, y_tr_ta, y_te_ta = train_test_split(
    X, y_TA, test_size=0.3, random_state=42
)

ta_results = []
for leaf in LEAF_TA_EC_SWEEP:
    pipe = make_pipe(depth=12, min_leaf=leaf)
    cv_s = cross_val_score(pipe, X, y_TA, cv=cv, scoring='r2', n_jobs=-1)
    pipe.fit(X_tr_ta, y_tr_ta)
    r2_tr = r2_score(y_tr_ta, pipe.predict(X_tr_ta))
    r2_te = r2_score(y_te_ta, pipe.predict(X_te_ta))
    gap   = r2_tr - r2_te
    ta_results.append({'leaf': leaf, 'cv': cv_s.mean(), 'std': cv_s.std(),
                       'r2_tr': r2_tr, 'r2_te': r2_te, 'gap': gap, 'pipe': pipe})
    print(f"  leaf={leaf}:  CV={cv_s.mean():.4f}±{cv_s.std():.4f}  gap={gap:.4f}")

best_ta = select_best(ta_results)
print(f"\n  → Best TA (min gap): leaf={best_ta['leaf']}  "
      f"CV={best_ta['cv']:.4f}  gap={best_ta['gap']:.4f}")


# =============================================================
# 17. TRAIN EC — LEAF SWEEP  (select by min gap)
# =============================================================
print("\n" + "=" * 60)
print("EC LEAF SWEEP  (max_depth=12, testing leaf=4/6/8)")
print("Phase 6: leaf=4 selected by CV → gap=0.0690")
print("Now selecting by MIN GAP")
print("=" * 60)

X_tr_ec, X_te_ec, y_tr_ec, y_te_ec = train_test_split(
    X, y_EC, test_size=0.3, random_state=42
)

ec_results = []
for leaf in LEAF_TA_EC_SWEEP:
    pipe = make_pipe(depth=12, min_leaf=leaf)
    cv_s = cross_val_score(pipe, X, y_EC, cv=cv, scoring='r2', n_jobs=-1)
    pipe.fit(X_tr_ec, y_tr_ec)
    r2_tr = r2_score(y_tr_ec, pipe.predict(X_tr_ec))
    r2_te = r2_score(y_te_ec, pipe.predict(X_te_ec))
    gap   = r2_tr - r2_te
    ec_results.append({'leaf': leaf, 'cv': cv_s.mean(), 'std': cv_s.std(),
                       'r2_tr': r2_tr, 'r2_te': r2_te, 'gap': gap, 'pipe': pipe})
    print(f"  leaf={leaf}:  CV={cv_s.mean():.4f}±{cv_s.std():.4f}  gap={gap:.4f}")

best_ec = select_best(ec_results)
print(f"\n  → Best EC (min gap): leaf={best_ec['leaf']}  "
      f"CV={best_ec['cv']:.4f}  gap={best_ec['gap']:.4f}")


# =============================================================
# 18. TRAIN DRP — FULL SWEEP
# Axes: depth (8/10) × leaf (6/8) × NWU_K (1/3)
# Selection: min gap (primary), max CV (tiebreak)
# =============================================================
print("\n" + "=" * 60)
print("DRP FULL SWEEP  (depth=8/10 × leaf=6/8 × NWU_K=1/3)")
print("Phase 6 best (by CV):  depth=10, leaf=6, K=3  gap=0.1035")
print("Phase 6 best (by gap): depth=8,  leaf=8, K=3  gap=0.0681")
print("Now selecting by MIN GAP across all 8 combos")
print("=" * 60)

X_tr_drp, X_te_drp, y_tr_drp, y_te_drp = train_test_split(
    X, y_DRP, test_size=0.3, random_state=42
)
X_tr_drp1, X_te_drp1, _, _ = train_test_split(
    X_drp1, y_DRP, test_size=0.3, random_state=42
)

drp_results = []
for nwu_k in NWU_K_DRP_SWEEP:
    X_tr_use = X_tr_drp1 if nwu_k == 1 else X_tr_drp
    X_te_use = X_te_drp1 if nwu_k == 1 else X_te_drp
    X_full   = X_drp1    if nwu_k == 1 else X

    for depth in DRP_DEPTH_SWEEP:
        for leaf in DRP_LEAF_SWEEP:
            pipe = make_pipe(depth=depth, min_leaf=leaf)
            cv_s = cross_val_score(pipe, X_full, y_DRP,
                                   cv=cv, scoring='r2', n_jobs=-1)
            pipe.fit(X_tr_use, y_tr_drp)
            r2_tr = r2_score(y_tr_drp, pipe.predict(X_tr_use))
            r2_te = r2_score(y_te_drp, pipe.predict(X_te_use))
            gap   = r2_tr - r2_te
            drp_results.append({
                'nwu_k': nwu_k, 'depth': depth, 'leaf': leaf,
                'cv': cv_s.mean(), 'std': cv_s.std(),
                'r2_tr': r2_tr, 'r2_te': r2_te, 'gap': gap,
                'pipe': pipe,
                'X_full': X_full,
                'X_val_key': 'k1' if nwu_k == 1 else 'k3',
            })
            print(f"  K={nwu_k}, depth={depth}, leaf={leaf}:  "
                  f"CV={cv_s.mean():.4f}±{cv_s.std():.4f}  gap={gap:.4f}")

best_drp = select_best(drp_results)
print(f"\n  → Best DRP (min gap): "
      f"K={best_drp['nwu_k']}, depth={best_drp['depth']}, leaf={best_drp['leaf']}  "
      f"CV={best_drp['cv']:.4f}  gap={best_drp['gap']:.4f}")


# =============================================================
# 19. LOCK MODELS
# =============================================================
models = {
    'Total Alkalinity':              best_ta['pipe'],
    'Electrical Conductance':        best_ec['pipe'],
    'Dissolved Reactive Phosphorus': best_drp['pipe'],
}


# =============================================================
# 20. FINAL SUMMARY
# =============================================================
summary_rows = [
    {
        'Parameter':      'Total Alkalinity',
        'best_leaf':      best_ta['leaf'],
        'KFold_CV_R2':    round(best_ta['cv'],   4),
        'CV_Std':         round(best_ta['std'],  4),
        'R2_Train':       round(best_ta['r2_tr'],4),
        'R2_Test':        round(best_ta['r2_te'],4),
        'Train_Test_Gap': round(best_ta['gap'],  4),
        'P6_Gap':         0.0713,
    },
    {
        'Parameter':      'Electrical Conductance',
        'best_leaf':      best_ec['leaf'],
        'KFold_CV_R2':    round(best_ec['cv'],   4),
        'CV_Std':         round(best_ec['std'],  4),
        'R2_Train':       round(best_ec['r2_tr'],4),
        'R2_Test':        round(best_ec['r2_te'],4),
        'Train_Test_Gap': round(best_ec['gap'],  4),
        'P6_Gap':         0.0690,
    },
    {
        'Parameter':      'Dissolved Reactive Phosphorus',
        'best_nwu_k':     best_drp['nwu_k'],
        'best_depth':     best_drp['depth'],
        'best_leaf':      best_drp['leaf'],
        'KFold_CV_R2':    round(best_drp['cv'],   4),
        'CV_Std':         round(best_drp['std'],  4),
        'R2_Train':       round(best_drp['r2_tr'],4),
        'R2_Test':        round(best_drp['r2_te'],4),
        'Train_Test_Gap': round(best_drp['gap'],  4),
        'P6_Gap':         0.0681,  # Phase 6 best-by-gap ref
    },
]

results_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 60)
print("FINAL SUMMARY  (P6_Gap = Phase 6 best-by-gap reference)")
print("=" * 60)
print(results_df[['Parameter','KFold_CV_R2','CV_Std',
                   'R2_Train','R2_Test','Train_Test_Gap','P6_Gap']].to_string(index=False))

print(f"\n  Gap deltas vs Phase 6 best-by-gap:")
for row in summary_rows:
    delta = row['Train_Test_Gap'] - row['P6_Gap']
    print(f"  {row['Parameter']:<35}  {delta:+.4f}")


# =============================================================
# 21. FEATURE IMPORTANCE
# =============================================================
print("\n" + "=" * 60)
print("FEATURE IMPORTANCES (top 20 per target)")
print("=" * 60)

importance_dfs = []
for name, pipe in models.items():
    rf_model   = pipe.named_steps['rf']
    feat_cols  = (FEATURE_COLS
                  if name != 'Dissolved Reactive Phosphorus'
                  or best_drp['nwu_k'] == 3
                  else FEATURE_COLS)
    imp        = pd.Series(rf_model.feature_importances_, index=feat_cols)
    imp_sorted = imp.sort_values(ascending=False)
    print(f"\n--- {name} ---")
    print(imp_sorted.head(20).round(4).to_string())
    imp_df = imp_sorted.reset_index()
    imp_df.columns = ['Feature', name]
    importance_dfs.append(imp_df.set_index('Feature'))

combined_imp = pd.concat(importance_dfs, axis=1).fillna(0)
combined_imp['Mean_Importance'] = combined_imp.mean(axis=1)
combined_imp = combined_imp.sort_values('Mean_Importance', ascending=False)

print("\n--- COMBINED ---")
print(combined_imp[['Mean_Importance']].round(4).to_string())

print(f"\n→ PAR feature importance:")
if 'glorich_PAR' in combined_imp.index:
    print(f"  glorich_PAR:  mean={combined_imp.loc['glorich_PAR','Mean_Importance']:.4f}  "
          f"DRP={combined_imp.loc['glorich_PAR','Dissolved Reactive Phosphorus']:.4f}")

print("\n→ Bottom 5 candidates to drop next iteration:")
print(combined_imp[['Mean_Importance']].tail(5).round(4).to_string())


# =============================================================
# 22. SUBMISSION
# =============================================================
# Determine which validation feature matrix to use for DRP
if best_drp['nwu_k'] == 1:
    X_val_drp = test_k1[FEATURE_COLS].copy().fillna(
        X_drp1.median()
    )
    print(f"\n  DRP submission uses K=1 NWU features")
else:
    X_val_drp = test_df2[FEATURE_COLS].copy().fillna(train_medians)
    print(f"\n  DRP submission uses K=3 NWU features")

X_val = test_df2[FEATURE_COLS].copy().fillna(train_medians)

submission_df = pd.DataFrame({
    'Longitude':                     test_df2['Longitude'],
    'Latitude':                      test_df2['Latitude'],
    'Sample Date':                   test_df2['Sample Date'],
    'Total Alkalinity':              models['Total Alkalinity'].predict(X_val),
    'Electrical Conductance':        models['Electrical Conductance'].predict(X_val),
    'Dissolved Reactive Phosphorus': models['Dissolved Reactive Phosphorus'].predict(X_val_drp),
})

fname = 'submission_rf_phase7.csv'
submission_df.to_csv(fname, index=False)

print(f"\n{'=' * 60}")
print(f"Submission saved → {fname}")
print(f"{'=' * 60}")
print(submission_df.head())
print(submission_df.describe().round(2))

LOADING DATA
Training rows:   9,319
Submission rows: 200

MERGING FEATURES — TRAINING DATA

[partner / flow features]
  Exact join: 2,391/9,319 rows matched — 6,928 rows need spatial fill
  After spatial fill: 903 rows still null (expected 0)

[rain features]
  Exact join: 100% match (9,319 rows)

[land cover features]

[Landsat features]

[TerraClimate features]

[NWU KD-tree context features  (K=3 base)]
  NWU KD-tree: 1553 unique stations

[Basin chemistry features]
  Basin model: 12 clusters over 1553 NWU stations

[WoSIS soil features]

[WoSIS soil features — building KD-trees]
  WoSIS clay: 1,214 surface locations
  WoSIS phaq: 1,221 surface locations

[GloRiCh river chemistry + PAR features]

[GloRiCh — filtering to South Africa and aggregating]
  SA observations: 416,837
  GloRiCh KD-tree: 1,304 unique SA stations

Final training shape: (9319, 62)

MERGING FEATURES — SUBMISSION DATA

[partner / flow features]
  Exact join: 0/200 rows matched — 200 rows need spatial fill
  After

In [ ]:
from google.colab import files
files.download("submission_rf_phase7.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Another new best of 0.468!**

This reinforces that the model is working, the hyperparameters are now as fine-tuned as they possibly can be, and that the only thing missing would be more high quality data. I will now make one final all-in model with some off the walls features and then record its results.

In [ ]:
# =============================================================
# RANDOM FOREST — PHASE 8 (FINAL)
# Built on Phase 7 (leaderboard: 0.468)
#
# HYPERPARAMETERS LOCKED (no sweeps — best from Phase 7):
#   TA:  max_depth=12, min_samples_leaf=8
#   EC:  max_depth=12, min_samples_leaf=8
#   DRP: max_depth=8,  min_samples_leaf=8, NWU_K=1
#
# BOTTOM-5 PRUNED FROM PHASE 7:
#   nwu_k5_DRP_std (0.0031), swir16 (0.0058),
#   ndmi_rain (0.0059), MNDWI (0.0063)
#
# NEW FEATURES ADDED (all verified on training correlations):
#
#   1. DRP Disagreement Index
#      drp_disagreement = |nwu_k1_DRP - glorich_DIP_idw|
#      r_DRP = +0.71 — strongest external DRP signal found.
#      Captures sites where local NWU chemistry and regional
#      GloRiCh DIP diverge. High disagreement = complex P
#      dynamics, likely agricultural hotspot or unusual
#      hydrology. Spatially transferable: both inputs are
#      available at all 200 validation locations.
#
#   2. EC Disagreement Index
#      ec_disagreement = |nwu_k1_EC - glorich_EC_idw|
#      r_EC = +0.88 — captures sites where local and regional
#      conductance differ, indicating geochemical complexity
#      or ionic concentration gradients.
#
#   3. Weathering Index
#      weathering_index = glorich_SOC / (glorich_Altitude + 1)
#      r_TA = -0.37. High SOC at low altitude = active organic
#      weathering and leaching → lower carbonate alkalinity.
#      Low SOC at high altitude = geochemical control → higher
#      alkalinity from mineral weathering.
#
#   4. Soil Phosphorus Binding Capacity
#      soil_P_binding = soil_clay_idw * soil_phaq_idw
#      r_DRP = +0.10. Clay × pH interaction: high clay at
#      moderate pH maximises P adsorption, binding P in soil
#      and reducing DRP in runoff.
#
# LOCKED (unchanged):
#   LOG_DRP_TARGET = False
#   N_BASIN_CLUSTERS = 12
#   KFold(n_splits=5, shuffle=True, random_state=42)
# =============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


# =============================================================
# CONFIGURATION — ALL LOCKED
# =============================================================
LOG_DRP_TARGET   = False
N_BASIN_CLUSTERS = 12
NWU_K_TA_EC      = 3
NWU_K_DRP        = 1
IDW_EPS          = 1e-6
WOSIS_K          = 5
GLORICH_K        = 5
SURFACE_DEPTH    = 30

TA_DEPTH  = 12;  TA_LEAF  = 8
EC_DEPTH  = 12;  EC_LEAF  = 8
DRP_DEPTH = 8;   DRP_LEAF = 8


# =============================================================
# 1. LOAD DATA
# =============================================================
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

base_df    = pd.read_csv('water_quality_training_dataset.csv')
partner_df = pd.read_csv('water_training_added_features.csv')
rain_df    = pd.read_csv('nasa_rain_features.csv')
lc_df      = pd.read_csv('esa_landcover_features.csv')
landsat_df = pd.read_csv('landsat_features_training.csv')
terra_df   = pd.read_csv('terraclimate_features_training.csv')
nwu_df     = pd.read_csv('nwu_water_quality_clean.csv')
test_df    = pd.read_csv('submission_template.csv')

clay_df    = pd.read_csv('wosis_latest_clay.csv')
orgc_df    = pd.read_csv('wosis_latest_orgc.csv')
phaq_df    = pd.read_csv('wosis_latest_phaq.csv')
glorich_df = pd.read_csv('glorich_cleaned_chemistry_dataset.csv')

print(f"Training rows:   {len(base_df):,}")
print(f"Submission rows: {len(test_df):,}")


# =============================================================
# 2. DATE NORMALISATION
# =============================================================
def normalise_date(df):
    df = df.copy()
    df['Sample Date'] = pd.to_datetime(
        df['Sample Date'], format='mixed', dayfirst=True
    ).dt.strftime('%Y-%m-%d')
    return df

base_df    = normalise_date(base_df)
partner_df = normalise_date(partner_df)
rain_df    = normalise_date(rain_df)
test_df    = normalise_date(test_df)
landsat_df = normalise_date(landsat_df)
terra_df   = normalise_date(terra_df)
nwu_df     = normalise_date(nwu_df)

JOIN_KEYS = ['Latitude', 'Longitude', 'Sample Date']


# =============================================================
# 3. SPATIAL FILL HELPER
# =============================================================
def spatial_fill(base, feature_df, join_keys=JOIN_KEYS):
    already_in_base = set(base.columns) - set(join_keys)
    feat_cols = [
        c for c in feature_df.columns
        if c not in join_keys and c not in already_in_base
    ]
    if not feat_cols:
        print("  No new feature columns to add — skipping merge.")
        return base

    feature_slim = feature_df[join_keys + feat_cols].copy()
    merged       = base.merge(feature_slim, on=join_keys, how='left')
    null_mask    = merged[feat_cols].isnull().any(axis=1)
    n_missing    = null_mask.sum()

    if n_missing == 0:
        print(f"  Exact join: 100% match ({len(base):,} rows)")
        return merged

    print(f"  Exact join: {len(base) - n_missing:,}/{len(base):,} rows matched "
          f"— {n_missing:,} rows need spatial fill")

    stat_medians = (
        feature_df
        .groupby(['Latitude', 'Longitude'])[feat_cols]
        .median()
        .reset_index()
    )
    tree           = cKDTree(stat_medians[['Latitude', 'Longitude']].values)
    missing_coords = base.loc[null_mask, ['Latitude', 'Longitude']].values
    _, nn_idxs     = tree.query(missing_coords, k=1)
    nearest_feats  = stat_medians[feat_cols].iloc[nn_idxs].reset_index(drop=True)
    merged_idx     = merged.index[null_mask]
    for col in feat_cols:
        merged.loc[merged_idx, col] = nearest_feats[col].values

    remaining_null = merged[feat_cols].isnull().any(axis=1).sum()
    print(f"  After spatial fill: {remaining_null} rows still null (expected 0)")
    return merged


# =============================================================
# 4. NWU FEATURES
# =============================================================
def build_nwu_tree(nwu_df):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            nwu_TA_med  = ('Total Alkalinity',              'median'),
            nwu_EC_med  = ('Electrical Conductance',        'median'),
            nwu_DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
        )
        .reset_index()
        .fillna(0)
    )
    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  NWU KD-tree: {len(stations)} unique stations")
    return stations, tree


def add_nwu_features(df, stations, tree, k):
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=k)

    if k == 1:
        dists = dists.reshape(-1, 1)
        idxs  = idxs.reshape(-1, 1)

    k1      = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    arr_TA  = stations['nwu_TA_med'].values[idxs]
    arr_EC  = stations['nwu_EC_med'].values[idxs]
    arr_DRP = stations['nwu_DRP_med'].values[idxs]

    weights    = 1.0 / (dists + IDW_EPS)
    weight_sum = weights.sum(axis=1, keepdims=True)
    w_norm     = weights / weight_sum

    result = df.copy()
    result['nwu_k1_TA']      = k1['nwu_TA_med'].values
    result['nwu_k1_EC']      = k1['nwu_EC_med'].values
    result['nwu_k1_DRP']     = k1['nwu_DRP_med'].values
    result['nwu_k5_TA_idw']  = (arr_TA  * w_norm).sum(axis=1)
    result['nwu_k5_EC_idw']  = (arr_EC  * w_norm).sum(axis=1)
    result['nwu_k5_DRP_idw'] = (arr_DRP * w_norm).sum(axis=1)
    return result


# =============================================================
# 5. BASIN CHEMISTRY FEATURES
# =============================================================
def build_basin_model(nwu_df, n_clusters=N_BASIN_CLUSTERS):
    stations = (
        nwu_df
        .groupby(['Latitude', 'Longitude'])
        .agg(
            TA_med  = ('Total Alkalinity',              'median'),
            EC_med  = ('Electrical Conductance',        'median'),
            DRP_med = ('Dissolved Reactive Phosphorus', 'median'),
        )
        .reset_index()
    )
    coords = stations[['Latitude', 'Longitude']].values
    km     = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    stations['basin'] = km.fit_predict(coords)

    basin_profiles = (
        stations
        .groupby('basin')
        .agg(
            basin_TA_med  = ('TA_med',  'median'),
            basin_EC_med  = ('EC_med',  'median'),
            basin_DRP_std = ('DRP_med', 'std'),
        )
        .reset_index()
        .fillna(0)
    )
    print(f"  Basin model: {n_clusters} clusters over {len(stations)} NWU stations")
    return stations, basin_profiles, km, coords


def add_basin_features(df, station_coords, basin_profiles, km):
    coords         = df[['Latitude', 'Longitude']].values
    basin_labels   = km.predict(coords)
    profile_lookup = basin_profiles.set_index('basin')
    row_profiles   = profile_lookup.loc[basin_labels].reset_index(drop=True)
    result         = df.copy()
    for col in profile_lookup.columns:
        result[col] = row_profiles[col].values
    return result


# =============================================================
# 6. WOSIS SOIL FEATURES
# =============================================================
def _build_wosis_tree(df, prop_name):
    surf = df[df['upper_depth'] <= SURFACE_DEPTH].copy()
    agg  = (
        surf
        .groupby(['X', 'Y'])['value_avg']
        .mean()
        .reset_index()
        .rename(columns={'value_avg': prop_name})
        .dropna(subset=[prop_name])
    )
    tree = cKDTree(agg[['Y', 'X']].values)
    print(f"  WoSIS {prop_name}: {len(agg):,} surface locations")
    return agg, tree


def build_wosis_trees(clay_df, orgc_df, phaq_df):
    print("\n[WoSIS soil features — building KD-trees]")
    return {
        'clay': _build_wosis_tree(clay_df, 'clay'),
        'phaq': _build_wosis_tree(phaq_df, 'phaq'),
    }


def add_wosis_features(df, wosis_trees):
    result = df.copy()
    for prop_name, (profiles, tree) in wosis_trees.items():
        coords      = result[['Latitude', 'Longitude']].values
        dists, idxs = tree.query(coords, k=WOSIS_K)
        values      = profiles[prop_name].values[idxs]
        weights     = 1.0 / (dists + IDW_EPS)
        w_norm      = weights / weights.sum(axis=1, keepdims=True)
        result[f'soil_{prop_name}_idw'] = (values * w_norm).sum(axis=1)
    return result


# =============================================================
# 7. GLORICH FEATURES + PAR
# =============================================================
def build_glorich_tree(glorich_df):
    print("\n[GloRiCh — filtering to South Africa and aggregating]")
    sa = glorich_df[glorich_df['Country'] == 'South Africa'].copy()
    print(f"  SA observations: {len(sa):,}")

    stations = (
        sa.groupby(['Latitude', 'Longitude'])
        .agg(
            glorich_TA_med      = ('Total_Alkalinity',      'median'),
            glorich_EC_med      = ('EC',                    'median'),
            glorich_EC_std      = ('EC',                    'std'),
            glorich_DIP_med     = ('Dissolved_Inorganic_P', 'median'),
            glorich_DIP_std     = ('Dissolved_Inorganic_P', 'std'),
            glorich_SOC         = ('SOC',                   'median'),
            glorich_Altitude    = ('Altitude',              'median'),
            glorich_Catch_Slope = ('Catch_Slope',           'median'),
            glorich_GLC_Forest  = ('GLC_Forest',            'median'),
        )
        .reset_index()
    )

    stations['glorich_PAR'] = (
        stations['glorich_DIP_med'] / (stations['glorich_TA_med'] + 1)
    )

    for col in stations.columns:
        if stations[col].isnull().any():
            stations[col] = stations[col].fillna(stations[col].median())

    tree = cKDTree(stations[['Latitude', 'Longitude']].values)
    print(f"  GloRiCh KD-tree: {len(stations):,} unique SA stations")
    return stations, tree


def add_glorich_features(df, stations, tree):
    coords      = df[['Latitude', 'Longitude']].values
    dists, idxs = tree.query(coords, k=GLORICH_K)
    result      = df.copy()

    for med_col, k1_col, idw_col, std_col in [
        ('glorich_TA_med',  'glorich_TA_k1',  'glorich_TA_idw',  None),
        ('glorich_EC_med',  'glorich_EC_k1',  'glorich_EC_idw',  'glorich_EC_std_feat'),
        ('glorich_DIP_med', 'glorich_DIP_k1', 'glorich_DIP_idw', 'glorich_DIP_std_feat'),
    ]:
        vals    = stations[med_col].values[idxs]
        weights = 1.0 / (dists + IDW_EPS)
        w_norm  = weights / weights.sum(axis=1, keepdims=True)
        result[k1_col]  = vals[:, 0]
        result[idw_col] = (vals * w_norm).sum(axis=1)
        if std_col:
            result[std_col] = vals.std(axis=1)

    nn = stations.iloc[idxs[:, 0]].reset_index(drop=True)
    for col in ['glorich_SOC', 'glorich_Altitude', 'glorich_Catch_Slope',
                'glorich_GLC_Forest', 'glorich_PAR']:
        result[col] = nn[col].values

    return result


# =============================================================
# 8. PHASE 8 NEW FEATURES
# Built after NWU and GloRiCh so all parent columns exist.
# =============================================================
def add_phase8_features(df):
    result = df.copy()

    # 1. DRP Disagreement Index — r_DRP = +0.71
    if 'nwu_k1_DRP' in result.columns and 'glorich_DIP_idw' in result.columns:
        result['drp_disagreement'] = (
            result['nwu_k1_DRP'] - result['glorich_DIP_idw']
        ).abs()

    # 2. EC Disagreement Index — r_EC = +0.88
    if 'nwu_k1_EC' in result.columns and 'glorich_EC_idw' in result.columns:
        result['ec_disagreement'] = (
            result['nwu_k1_EC'] - result['glorich_EC_idw']
        ).abs()

    # 3. Weathering Index — r_TA = -0.37
    if 'glorich_SOC' in result.columns and 'glorich_Altitude' in result.columns:
        result['weathering_index'] = (
            result['glorich_SOC'] / (result['glorich_Altitude'] + 1)
        )

    # 4. Soil Phosphorus Binding Capacity — r_DRP = +0.10
    if 'soil_clay_idw' in result.columns and 'soil_phaq_idw' in result.columns:
        result['soil_P_binding'] = (
            result['soil_clay_idw'] * result['soil_phaq_idw']
        )

    return result


# =============================================================
# 9. MERGE ALL FEATURES — TRAINING
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — TRAINING DATA")
print("=" * 60)

print("\n[partner / flow features]")
train_df = spatial_fill(base_df, partner_df)

print("\n[rain features]")
train_df = spatial_fill(train_df, rain_df)

print("\n[land cover features]")
train_df = train_df.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
train_df = train_df.merge(landsat_df, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
train_df = train_df.merge(terra_df, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features  (K=3)]")
nwu_stations, nwu_tree = build_nwu_tree(nwu_df)
train_df = add_nwu_features(train_df, nwu_stations, nwu_tree, k=NWU_K_TA_EC)

print("\n[Basin chemistry features]")
station_df, basin_profiles, km_basin, station_coords = build_basin_model(nwu_df)
train_df = add_basin_features(train_df, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
wosis_trees = build_wosis_trees(clay_df, orgc_df, phaq_df)
train_df = add_wosis_features(train_df, wosis_trees)

print("\n[GloRiCh river chemistry + PAR features]")
glorich_stations, glorich_tree = build_glorich_tree(glorich_df)
train_df = add_glorich_features(train_df, glorich_stations, glorich_tree)

print("\n[Phase 8 new features]")
train_df = add_phase8_features(train_df)

print(f"\nFinal training shape: {train_df.shape}")


# =============================================================
# 10. MERGE ALL FEATURES — SUBMISSION
# =============================================================
print("\n" + "=" * 60)
print("MERGING FEATURES — SUBMISSION DATA")
print("=" * 60)

landsat_val = normalise_date(pd.read_csv('landsat_features_validation.csv'))
terra_val   = normalise_date(pd.read_csv('terraclimate_features_validation.csv'))

print("\n[partner / flow features]")
test_df2 = spatial_fill(test_df, partner_df)

print("\n[rain features]")
test_df2 = spatial_fill(test_df2, rain_df)

print("\n[land cover features]")
test_df2 = test_df2.merge(lc_df, on=['Latitude', 'Longitude'], how='left')

print("\n[Landsat features]")
test_df2 = test_df2.merge(landsat_val, on=JOIN_KEYS, how='left')

print("\n[TerraClimate features]")
test_df2 = test_df2.merge(terra_val, on=JOIN_KEYS, how='left')

print("\n[NWU KD-tree context features  (K=3)]")
test_df2 = add_nwu_features(test_df2, nwu_stations, nwu_tree, k=NWU_K_TA_EC)

print("\n[Basin chemistry features]")
test_df2 = add_basin_features(test_df2, station_coords, basin_profiles, km_basin)

print("\n[WoSIS soil features]")
test_df2 = add_wosis_features(test_df2, wosis_trees)

print("\n[GloRiCh river chemistry + PAR features]")
test_df2 = add_glorich_features(test_df2, glorich_stations, glorich_tree)

print("\n[Phase 8 new features]")
test_df2 = add_phase8_features(test_df2)

# K=1 NWU variant for DRP — built after all other features
# so disagreement index uses the correct K=1 DRP anchor
print("\n[Building K=1 NWU features for DRP model]")
train_k1 = add_nwu_features(train_df, nwu_stations, nwu_tree, k=NWU_K_DRP)
test_k1  = add_nwu_features(test_df2, nwu_stations, nwu_tree, k=NWU_K_DRP)
train_k1 = add_phase8_features(train_k1)
test_k1  = add_phase8_features(test_k1)

print(f"\nFinal submission shape: {test_df2.shape}")


# =============================================================
# 11. TEMPORAL FEATURES
# =============================================================
def add_temporal(df):
    df    = df.copy()
    dates = pd.to_datetime(df['Sample Date'])
    df['DayOfYear']            = dates.dt.dayofyear
    df['Month']                = dates.dt.month
    df['days_since_wet_start'] = (df['DayOfYear'] - 274) % 365
    return df

train_df = add_temporal(train_df)
test_df2 = add_temporal(test_df2)
train_k1 = add_temporal(train_k1)
test_k1  = add_temporal(test_k1)


# =============================================================
# 12. PHYSICS-BASED INTERACTION + PULSE FEATURES
# =============================================================
def add_interactions(df):
    df = df.copy()
    if 'rain_30d_sum' in df.columns and 'temp_7d_mean' in df.columns:
        df['rain_temp_30d'] = df['rain_30d_sum'] * df['temp_7d_mean']
    if 'rain_30d_mean' in df.columns and 'rh_7d_mean' in df.columns:
        df['rain_rh_30d'] = df['rain_30d_mean'] * df['rh_7d_mean']
    if 'rain_30d_sum' in df.columns and 'rain_7d_mean' in df.columns:
        df['rain_pulse_ratio'] = df['rain_7d_mean'] / (df['rain_30d_sum'] / 30 + 1e-6)
    if 'pet' in df.columns and 'rain_30d_mean' in df.columns:
        df['pet_rain_ratio'] = df['pet'] / (df['rain_30d_mean'] + 1e-6)
    # ndmi_rain PRUNED Phase 8 (0.0059)
    return df

def add_rainfall_pulse_features(df):
    df = df.copy()
    if 'rain_7d_mean' in df.columns and 'rain_30d_mean' in df.columns:
        df['rain_dry_spell_ratio'] = (
            df['rain_30d_mean'] / (df['rain_7d_mean'] + 1e-6)
        ).clip(upper=50)
    return df

for _df_name in ['train_df', 'test_df2', 'train_k1', 'test_k1']:
    exec(f"{_df_name} = add_interactions({_df_name})")
    exec(f"{_df_name} = add_rainfall_pulse_features({_df_name})")


# =============================================================
# 13. FEATURE COLUMNS
# =============================================================
BASELINE_FEATURES = [
    'Longitude', 'Latitude',
    'rh_7d_mean', 'distance_to_station_km',
    'rain_30d_mean', 'rain_30d_sum',
    'temp_7d_mean',
]

EY_FEATURES = [
    'swir22', 'pet',
    # swir16  PRUNED (0.0058)
    # nir     PRUNED Phase 7 (0.0064)
    # MNDWI   PRUNED (0.0063)
    # NDMI    PRUNED (standalone weak; ndmi_rain also pruned)
]

TEMPORAL_FEATURES = [
    'DayOfYear',
    'days_since_wet_start',
]

INTERACTION_FEATURES = [
    'rain_temp_30d', 'rain_rh_30d', 'rain_pulse_ratio',
    'pet_rain_ratio',
    # ndmi_rain PRUNED (0.0059)
]

PULSE_FEATURES = [
    'rain_dry_spell_ratio',
]

NWU_FEATURES = [
    'nwu_k1_TA',  'nwu_k1_EC',  'nwu_k1_DRP',
    'nwu_k5_TA_idw', 'nwu_k5_EC_idw', 'nwu_k5_DRP_idw',
    # nwu_k5_DRP_std PRUNED (0.0031)
]

BASIN_FEATURES = [
    'basin_TA_med',
    'basin_EC_med',
    'basin_DRP_std',
]

WOSIS_FEATURES = [
    'soil_clay_idw',
    'soil_phaq_idw',
]

GLORICH_FEATURES = [
    'glorich_TA_k1',  'glorich_TA_idw',
    'glorich_EC_k1',  'glorich_EC_idw',  'glorich_EC_std_feat',
    'glorich_DIP_k1', 'glorich_DIP_idw', 'glorich_DIP_std_feat',
    'glorich_SOC',    'glorich_Altitude',
    'glorich_Catch_Slope', 'glorich_GLC_Forest',
    'glorich_PAR',
]

PHASE8_FEATURES = [
    'drp_disagreement',
    'ec_disagreement',
    'weathering_index',
    'soil_P_binding',
]

FEATURE_COLS = [
    c for c in
    BASELINE_FEATURES + EY_FEATURES + TEMPORAL_FEATURES
    + INTERACTION_FEATURES + PULSE_FEATURES
    + NWU_FEATURES + BASIN_FEATURES
    + WOSIS_FEATURES + GLORICH_FEATURES
    + PHASE8_FEATURES
    if c in train_df.columns
]

print(f"\n{'=' * 60}")
print("FEATURE SUMMARY")
print(f"{'=' * 60}")
print(f"Total features: {len(FEATURE_COLS)}")
print(f"  Baseline:       {sum(c in train_df.columns for c in BASELINE_FEATURES)}")
print(f"  EY satellite:   {sum(c in train_df.columns for c in EY_FEATURES)}")
print(f"  Temporal:       {sum(c in train_df.columns for c in TEMPORAL_FEATURES)}")
print(f"  Interaction:    {sum(c in train_df.columns for c in INTERACTION_FEATURES)}")
print(f"  Pulse:          {sum(c in train_df.columns for c in PULSE_FEATURES)}")
print(f"  NWU context:    {sum(c in train_df.columns for c in NWU_FEATURES)}  (TA/EC K=3, DRP K=1)")
print(f"  Basin:          {sum(c in train_df.columns for c in BASIN_FEATURES)}")
print(f"  WoSIS soil:     {sum(c in train_df.columns for c in WOSIS_FEATURES)}")
print(f"  GloRiCh:        {sum(c in train_df.columns for c in GLORICH_FEATURES)}")
print(f"  Phase 8 (new):  {sum(c in train_df.columns for c in PHASE8_FEATURES)}")


# =============================================================
# 14. PREPARE DATA
# =============================================================
def resolve_target_col(df, col):
    if col in df.columns:        return df[col]
    if col + '_x' in df.columns: return df[col + '_x']
    raise KeyError(f"Target column '{col}' not found.")

X      = train_df[FEATURE_COLS].copy().fillna(train_df[FEATURE_COLS].median())
X_drp1 = train_k1[FEATURE_COLS].copy().fillna(train_k1[FEATURE_COLS].median())

train_medians      = X.median()
train_medians_drp1 = X_drp1.median()

y_TA  = resolve_target_col(train_df, 'Total Alkalinity')
y_EC  = resolve_target_col(train_df, 'Electrical Conductance')
y_DRP = resolve_target_col(train_df, 'Dissolved Reactive Phosphorus')

cv = KFold(n_splits=5, shuffle=True, random_state=42)


# =============================================================
# 15. MODEL FACTORY
# =============================================================
def make_pipe(depth, min_leaf):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            n_estimators     = 500,
            max_depth        = depth,
            min_samples_leaf = min_leaf,
            max_features     = 'sqrt',
            bootstrap        = True,
            random_state     = 42,
            n_jobs           = -1,
        ))
    ])


# =============================================================
# 16. TRAIN — LOCKED HYPERPARAMETERS
# =============================================================
models  = {}
results = []

configs = [
    ('Total Alkalinity',              y_TA,  X,      TA_DEPTH,  TA_LEAF,  0.8496, 0.0552),
    ('Electrical Conductance',        y_EC,  X,      EC_DEPTH,  EC_LEAF,  0.8524, 0.0519),
    ('Dissolved Reactive Phosphorus', y_DRP, X_drp1, DRP_DEPTH, DRP_LEAF, 0.6017, 0.0690),
]

for name, y_target, X_use, depth, leaf, p7_cv, p7_gap in configs:
    is_drp  = 'Phosphorus' in name
    nwu_tag = 'NWU_K=1' if is_drp else 'NWU_K=3'

    print("\n" + "=" * 60)
    print(f"Training: {name}")
    print(f"  Config: max_depth={depth}, min_samples_leaf={leaf}, {nwu_tag}")
    print(f"  Phase 7 ref: CV={p7_cv:.4f}, gap={p7_gap:.4f}")
    print("=" * 60)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_use, y_target, test_size=0.3, random_state=42
    )

    pipe      = make_pipe(depth, leaf)
    cv_scores = cross_val_score(pipe, X_use, y_target, cv=cv,
                                scoring='r2', n_jobs=-1)
    pipe.fit(X_tr, y_tr)

    r2_train = r2_score(y_tr, pipe.predict(X_tr))
    r2_test  = r2_score(y_te, pipe.predict(X_te))
    gap      = r2_train - r2_test
    rmse     = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))

    delta_cv  = cv_scores.mean() - p7_cv
    delta_gap = gap - p7_gap

    print(f"  CV R²:          {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  "
          f"(Δ vs P7: {delta_cv:+.4f})")
    print(f"  Train R²:       {r2_train:.4f}")
    print(f"  Test  R²:       {r2_test:.4f}")
    print(f"  Gap:            {gap:.4f}  (Δ vs P7: {delta_gap:+.4f})")
    print(f"  RMSE:           {rmse:.4f}")

    models[name] = pipe
    results.append({
        'Parameter':      name,
        'KFold_CV_R2':    round(cv_scores.mean(), 4),
        'CV_Std':         round(cv_scores.std(),  4),
        'R2_Train':       round(r2_train, 4),
        'R2_Test':        round(r2_test,  4),
        'Gap':            round(gap, 4),
        'P7_Gap':         p7_gap,
        'Gap_Delta':      round(gap - p7_gap, 4),
        'RMSE':           round(rmse, 4),
    })


# =============================================================
# 17. FINAL SUMMARY
# =============================================================
results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("FINAL SUMMARY  (Gap_Delta = improvement vs Phase 7)")
print("=" * 60)
print(results_df[['Parameter','KFold_CV_R2','CV_Std',
                   'R2_Train','R2_Test','Gap','P7_Gap','Gap_Delta']].to_string(index=False))

all_gaps_improved = all(r['Gap_Delta'] <= 0 for r in results)
print(f"\n  All gaps improved vs Phase 7: {all_gaps_improved}")
print(f"  Phase 7 refs — TA gap: 0.0552 | EC gap: 0.0519 | DRP gap: 0.0690")


# =============================================================
# 18. FEATURE IMPORTANCE
# =============================================================
print("\n" + "=" * 60)
print("FEATURE IMPORTANCES (top 20 per target)")
print("=" * 60)

importance_dfs = []

for name, pipe in models.items():
    rf_model   = pipe.named_steps['rf']
    imp        = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
    imp_sorted = imp.sort_values(ascending=False)

    print(f"\n--- {name} ---")
    print(imp_sorted.head(20).round(4).to_string())

    imp_df         = imp_sorted.reset_index()
    imp_df.columns = ['Feature', name]
    importance_dfs.append(imp_df.set_index('Feature'))

combined_imp = pd.concat(importance_dfs, axis=1).fillna(0)
combined_imp['Mean_Importance'] = combined_imp.mean(axis=1)
combined_imp = combined_imp.sort_values('Mean_Importance', ascending=False)

print("\n--- COMBINED (mean importance across all targets) ---")
print(combined_imp[['Mean_Importance']].round(4).to_string())

print("\n→ Phase 8 new feature importances:")
for f in PHASE8_FEATURES:
    if f in combined_imp.index:
        mean_imp = combined_imp.loc[f, 'Mean_Importance']
        drp_imp  = combined_imp.loc[f, 'Dissolved Reactive Phosphorus']
        ta_imp   = combined_imp.loc[f, 'Total Alkalinity']
        ec_imp   = combined_imp.loc[f, 'Electrical Conductance']
        print(f"  {f:<25}  mean={mean_imp:.4f}  "
              f"TA={ta_imp:.4f}  EC={ec_imp:.4f}  DRP={drp_imp:.4f}")

print("\n→ PAR feature importance:")
if 'glorich_PAR' in combined_imp.index:
    print(f"  glorich_PAR:  "
          f"mean={combined_imp.loc['glorich_PAR', 'Mean_Importance']:.4f}  "
          f"DRP={combined_imp.loc['glorich_PAR', 'Dissolved Reactive Phosphorus']:.4f}")

print("\n→ Bottom 5 candidates to drop next iteration:")
print(combined_imp[['Mean_Importance']].tail(5).round(4).to_string())


# =============================================================
# 19. SUBMISSION
# =============================================================
X_val_ta_ec = test_df2[FEATURE_COLS].copy().fillna(train_medians)
X_val_drp   = test_k1[FEATURE_COLS].copy().fillna(train_medians_drp1)

print(f"\n  TA/EC predictions use K=3 NWU features")
print(f"  DRP predictions use  K=1 NWU features")

submission_df = pd.DataFrame({
    'Longitude':                     test_df2['Longitude'],
    'Latitude':                      test_df2['Latitude'],
    'Sample Date':                   test_df2['Sample Date'],
    'Total Alkalinity':              models['Total Alkalinity'].predict(X_val_ta_ec),
    'Electrical Conductance':        models['Electrical Conductance'].predict(X_val_ta_ec),
    'Dissolved Reactive Phosphorus': models['Dissolved Reactive Phosphorus'].predict(X_val_drp),
})

fname = 'submission_rf_phase8.csv'
submission_df.to_csv(fname, index=False)

print(f"\n{'=' * 60}")
print(f"Submission saved → {fname}")
print(f"{'=' * 60}")
print(submission_df.head())
print(submission_df.describe().round(2))

LOADING DATA
Training rows:   9,319
Submission rows: 200

MERGING FEATURES — TRAINING DATA

[partner / flow features]
  Exact join: 2,391/9,319 rows matched — 6,928 rows need spatial fill
  After spatial fill: 903 rows still null (expected 0)

[rain features]
  Exact join: 100% match (9,319 rows)

[land cover features]

[Landsat features]

[TerraClimate features]

[NWU KD-tree context features  (K=3)]
  NWU KD-tree: 1553 unique stations

[Basin chemistry features]
  Basin model: 12 clusters over 1553 NWU stations

[WoSIS soil features]

[WoSIS soil features — building KD-trees]
  WoSIS clay: 1,214 surface locations
  WoSIS phaq: 1,221 surface locations

[GloRiCh river chemistry + PAR features]

[GloRiCh — filtering to South Africa and aggregating]
  SA observations: 416,837
  GloRiCh KD-tree: 1,304 unique SA stations

[Phase 8 new features]

Final training shape: (9319, 65)

MERGING FEATURES — SUBMISSION DATA

[partner / flow features]
  Exact join: 0/200 rows matched — 200 rows need s

In [ ]:
from google.colab import files
files.download("submission_rf_phase8.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Boosted us slightly to 0.4689**